In [ ]:
# CONFIGURAÇÃO DO AMBIENTE 

In [ ]:
import os
import sys
import json
import logging
import warnings
import hashlib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm
import seaborn as sns


from scipy import stats
from scipy.stats import (
    ttest_ind, f_oneway, kruskal, chi2_contingency,
    mannwhitneyu, shapiro, levene, pearsonr, spearmanr, norm, kstest, probplot
)

import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import MultiComparison
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.stats.anova import anova_lm
from statsmodels.regression.linear_model import GLSAR

import geopandas as gpd
import fiona 

from sklearn.preprocessing import StandardScaler

from IPython.display import display, Markdown, HTML

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Configurar pandas para exibição completa de colunas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
pd.set_option('display.max_rows', 100)

CODIGOS_BARIATRICOS = {
    '407010173': 'Derivação Gástrica (Bypass)',
    '407010181': 'Banda Gástrica Ajustável',
    '407010386': 'Cirurgia Bariátrica Videolaparoscópica',
    '407010360': 'Gastrectomia Vertical (Manga)'
}

CORES_PROCEDIMENTOS = {
    '407010173': '#e74c3c',  # Vermelho
    '407010181': '#3498db',  # Azul
    '407010386': '#2ecc71',  # Verde
    '407010360': '#f39c12'   # Laranja
}

NOMES_CURTOS = {
    '407010173': 'Derivação',
    '407010181': 'Banda',
    '407010386': 'Videolaparoscopia',
    '407010360': 'Manga'
}

In [ ]:
# Criar logs
LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(
            filename=os.path.join(LOG_DIR, f"analise_bariatrica_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"),
            encoding='utf-8'
        ),
        logging.StreamHandler(sys.stdout)
    ]
)

logger = logging.getLogger("BariatricAnalysis")
logger.info("Sistema de logging inicializado")

In [ ]:
# Configurações de visualização
sns.set_style("whitegrid")

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.dpi'] = 600
plt.rcParams['savefig.dpi'] = 600
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.format'] = 'png'

# Paleta de cores
colors = sns.color_palette("Set2", 8)
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=colors)

# Suprimir avisos
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

logger.info("Configurações de visualização concluidas")

In [ ]:
# Definir diretórios
DIR_RESULTADOS = "resultados_analise_bariatrica"
DIR_TABELAS = os.path.join(DIR_RESULTADOS, "tabelas")
DIR_GRAFICOS = os.path.join(DIR_RESULTADOS, "graficos")
DIR_MAPAS = os.path.join(DIR_RESULTADOS, "mapas")
DIR_DADOS = os.path.join(DIR_RESULTADOS, "dados_processados")
DIR_LOGS = os.path.join(DIR_RESULTADOS, "logs")

# Criar estrutura de diretórios
for directory in [DIR_RESULTADOS, DIR_TABELAS, DIR_GRAFICOS, DIR_MAPAS, DIR_DADOS, DIR_LOGS]:
    os.makedirs(directory, exist_ok=True)

logger.info(f"Estrutura de diretórios criada em: {DIR_RESULTADOS}")

In [ ]:
# ETL

In [ ]:
# Caminho para o arquivo
CAMINHO_DADOS = " "  #caminho para a data

# Período de estudo
ANO_INICIO = 2016
ANO_FIM = 2025


# Variáveis críticas para análise
VAR_DEMOGRAFICAS = ['IDADE', 'COD_IDADE', 'SEXO', 'ESTADO_ORIGEM']
VAR_CLINICAS = ['VAL_TOT', 'DIAS_PERM', 'PROC_REA', 'ANO_CMPT', 'MES_CMPT']
VAR_IDENT = ['N_AIH', 'IDENT']

logger.info("Parâmetros de ETL definidos")

In [ ]:
def carregar_dados_datasus(caminho):
    encodings = ['latin-1', 'ISO-8859-1', 'utf-8', 'cp1252']
    
    for enc in encodings:
        try:
            df = pd.read_csv(caminho, encoding=enc, low_memory=False)
            logger.info(f"Arquivo carregado com encoding: {enc}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError("Encoding não compatível identificado")

# Carregar dataset
df = carregar_dados_datasus(CAMINHO_DADOS)
display(Markdown(f"**Registros brutos carregados:** {len(df):,}"))
display(Markdown(f"**Colunas disponíveis:** {len(df.columns)}"))


In [ ]:
# Verificar tipo atual
print(f"Tipo atual de PROC_REA: {df['PROC_REA'].dtype}")

# Converter para string
df['PROC_REA'] = df['PROC_REA'].astype(str).str.strip()

# Verificar conversão
print(f"Tipo após conversão: {df['PROC_REA'].dtype}")
print(f"Exemplos de valores: {df['PROC_REA'].head()}")

In [ ]:
# Filtrar procedimentos
df = df[df['PROC_REA'].isin(CODIGOS_BARIATRICOS.keys())].copy()
display(Markdown(f"**Registros após filtro de procedimento:** {len(df):,}"))

# Adicionar nome do procedimento
df['NOME_PROCEDIMENTO'] = df['PROC_REA'].map(CODIGOS_BARIATRICOS)

In [ ]:
#TEMPO
df = df[(df['ANO_CMPT'] >= ANO_INICIO) & (df['ANO_CMPT'] <= ANO_FIM)].copy()
display(Markdown(f"**Registros após filtro temporal ({ANO_INICIO}-{ANO_FIM}):** {len(df):,}"))


In [ ]:
# IDADE 
df['IDADE'] = pd.to_numeric(df['IDADE'], errors='coerce')
df = df[(df['IDADE'] >= 0) & (df['IDADE'] <= 120)].copy()


In [ ]:
# SEXO
df['SEXO'] = pd.to_numeric(df['SEXO'], errors='coerce')
df['SEXO_DESC'] = df['SEXO'].map({
    1: 'Masculino',
    2: 'Feminino',
    3: 'Feminino',
    0: 'Ignorado',
    9: 'Ignorado'
}).fillna('Ignorado')

In [ ]:
#UF DE RESIDÊNCIA
df['UF_SIGLA'] = df['ESTADO_ORIGEM'].astype(str).str.strip().str.upper()

# Validar siglas IBGE
siglas_validas = ['AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS',
                  'MG','PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR',
                  'SC','SP','SE','TO']
df = df[df['UF_SIGLA'].isin(siglas_validas)].copy()

In [ ]:
regioes = {
    'Norte': ['RO','AC','AM','RR','PA','AP','TO'],
    'Nordeste': ['MA','PI','CE','RN','PB','PE','AL','SE','BA'],
    'Centro-Oeste': ['MS','MT','GO','DF'],
    'Sudeste': ['SP','RJ','MG','ES'],
    'Sul': ['PR','SC','RS']
}
df['REGIAO'] = df['UF_SIGLA'].apply(
    lambda uf: next((r for r, estados in regioes.items() if uf in estados), 'Desconhecida')
)

In [ ]:
#VALOR TOTAL
df['VAL_TOT'] = pd.to_numeric(df['VAL_TOT'], errors='coerce')
df = df[(df['VAL_TOT'] > 0) & (df['VAL_TOT'].notna())].copy()

# DIAS DE PERMANÊNCIA
df['DIAS_PERM'] = pd.to_numeric(df['DIAS_PERM'], errors='coerce')
df = df[(df['DIAS_PERM'] >= 0) & (df['DIAS_PERM'] <= 90)].copy()

In [ ]:
# RELATÓRIO DE MISSING DATA
vars_criticas = ['IDADE', 'SEXO_DESC', 'UF_SIGLA', 'REGIAO', 'VAL_TOT', 'DIAS_PERM', 'ANO_CMPT']

missing_report = pd.DataFrame({
    'Variavel': vars_criticas,
    'N_Missing': [df[col].isna().sum() for col in vars_criticas if col in df.columns],
    'Percentual': [df[col].isna().mean() * 100 for col in vars_criticas if col in df.columns]
}).sort_values('Percentual', ascending=False)

display(Markdown("## Dados Ausentes"))
display(missing_report.style.format({'N_Missing': '{:,.0f}', 'Percentual': '{:.2f}%'}))

In [ ]:
#EXPORTAÇÃO DO DATASET LIMPO
DIR_RESULTADOS = "resultados_analise_bariatrica"
DIR_DADOS = os.path.join(DIR_RESULTADOS, "dados_processados")
os.makedirs(DIR_DADOS, exist_ok=True)

# Exportar em Parquet e CSV
df.to_parquet(os.path.join(DIR_DADOS, "dataset_limpo.parquet"), index=False)
df.to_csv(os.path.join(DIR_DADOS, "dataset_limpo.csv"), index=False, encoding='utf-8')

display(Markdown(f"**etl Concluído** | {len(df):,} registros prontos para análise"))
display(Markdown(f"**Período:** {df['ANO_CMPT'].min():.0f} - {df['ANO_CMPT'].max():.0f}"))
display(Markdown(f"**Procedimentos:** {df['NOME_PROCEDIMENTO'].nunique()} tipos"))
display(Markdown(f"**UFs cobertas:** {df['UF_SIGLA'].nunique()}/27"))

In [ ]:
# ANÁLISE DESCRITIVA EPIDEMIOLÓGICA

In [ ]:
# Carregar dataset
CAMINHO_DATASET_LIMPO = os.path.join(DIR_DADOS, "dataset_limpo.parquet")
df = pd.read_parquet(CAMINHO_DATASET_LIMPO)

logger.info(f"Dataset carregado para análise descritiva: {len(df):,} registros")
display(Markdown(f"**Registros para análise descritiva:** {len(df):,}"))

In [ ]:
def calcular_ic95_wilson(n, p, nivel_confianca=0.95):
    """
    Calcula Intervalo de Confiança de 95% usando Wilson Score Interval
    """    
    z = norm.ppf(1 - (1 - nivel_confianca) / 2)
    denominador = 1 + z**2 / n
    centro = (p + z**2 / (2 * n)) / denominador
    margem = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denominador
    
    return centro - margem, centro + margem

logger.info("Função de IC95% (Wilson) carregada")

In [ ]:
# Variáveis demográficas e clínicas
VAR_DESCRITIVAS = {
    'IDADE': 'Idade (anos)',
    'SEXO_DESC': 'Sexo',
    'UF_SIGLA': 'UF de Residência',
    'REGIAO': 'Região',
    'NOME_PROCEDIMENTO': 'Tipo de Procedimento',
    'VAL_TOT': 'Custo Total (R$)',
    'DIAS_PERM': 'Dias de Permanência',
    'ANO_CMPT': 'Ano de Competência'
}

display(Markdown("##Caracterização Geral da Amostra"))
display(Markdown(f"**Período do estudo:** {df['ANO_CMPT'].min():.0f} - {df['ANO_CMPT'].max():.0f}"))
display(Markdown(f"**Total de procedimentos:** {len(df):,}"))

In [ ]:
# Tabela de distribuição de procedimentos
dist_procedimentos = df['NOME_PROCEDIMENTO'].value_counts()
dist_procedimentos_pct = df['NOME_PROCEDIMENTO'].value_counts(normalize=True) * 100

tabela_procedimentos = pd.DataFrame({
    'Procedimento': dist_procedimentos.index,
    'N': dist_procedimentos.values,
    'Percentual (%)': dist_procedimentos_pct.values
})

# Calcular IC95% para cada proporção
tabela_procedimentos['IC95%_Inferior'] = 0.0
tabela_procedimentos['IC95%_Superior'] = 0.0

for idx, row in tabela_procedimentos.iterrows():
    n = len(df)
    p = row['Percentual (%)'] / 100
    ic_inf, ic_sup = calcular_ic95_wilson(n, p)
    tabela_procedimentos.loc[idx, 'IC95%_Inferior'] = ic_inf * 100
    tabela_procedimentos.loc[idx, 'IC95%_Superior'] = ic_sup * 100

display(Markdown("### 1.1 Distribuição por Tipo de Procedimento"))
display(tabela_procedimentos.style.format({
    'N': '{:,.0f}',
    'Percentual (%)': '{:.2f}%',
    'IC95%_Inferior': '{:.2f}%',
    'IC95%_Superior': '{:.2f}%'
}).background_gradient(subset=['N'], cmap='Blues'))

# Salvar tabela
tabela_procedimentos.to_csv(
    os.path.join(DIR_TABELAS, "distribuicao_procedimentos.csv"),
    index=False,
    encoding='utf-8'
)

In [ ]:
display(Markdown("Características Demográficas"))

# IDADE
idade_stats = {
    'N': df['IDADE'].notna().sum(),
    'Média': df['IDADE'].mean(),
    'Mediana': df['IDADE'].median(),
    'Desvio Padrão': df['IDADE'].std(),
    'Mínimo': df['IDADE'].min(),
    'Máximo': df['IDADE'].max(),
    'Q1 (25%)': df['IDADE'].quantile(0.25),
    'Q3 (75%)': df['IDADE'].quantile(0.75),
    'IIQ': df['IDADE'].quantile(0.75) - df['IDADE'].quantile(0.25)
}

tabela_idade = pd.DataFrame([idade_stats])
tabela_idade = tabela_idade.rename(columns={
    'N': 'N Válidos', 'IIQ': 'Intervalo Interquartil'
})

display(Markdown("Idade (anos)"))
display(tabela_idade.style.format({
    'N Válidos': '{:,.0f}',
    'Média': '{:.2f}',
    'Mediana': '{:.2f}',
    'Desvio Padrão': '{:.2f}',
    'Mínimo': '{:.1f}',
    'Máximo': '{:.1f}',
    'Q1 (25%)': '{:.1f}',
    'Q3 (75%)': '{:.1f}',
    'Intervalo Interquartil': '{:.2f}'
}))

# SEXO
if 'SEXO_DESC' in df.columns:
    dist_sexo = df['SEXO_DESC'].value_counts()
    dist_sexo_pct = df['SEXO_DESC'].value_counts(normalize=True) * 100
    
    tabela_sexo = pd.DataFrame({
        'Sexo': dist_sexo.index,
        'N': dist_sexo.values,
        'Percentual (%)': dist_sexo_pct.values
    })
    
    # Calcular IC95%
    tabela_sexo['IC95%_Inferior'] = 0.0
    tabela_sexo['IC95%_Superior'] = 0.0
    
    for idx, row in tabela_sexo.iterrows():
        n = len(df)
        p = row['Percentual (%)'] / 100
        ic_inf, ic_sup = calcular_ic95_wilson(n, p)
        tabela_sexo.loc[idx, 'IC95%_Inferior'] = ic_inf * 100
        tabela_sexo.loc[idx, 'IC95%_Superior'] = ic_sup * 100
    
    display(Markdown("**Sexo**"))
    display(tabela_sexo.style.format({
        'N': '{:,.0f}',
        'Percentual (%)': '{:.2f}%',
        'IC95%_Inferior': '{:.2f}%',
        'IC95%_Superior': '{:.2f}%'
    }))
    
    # Salvar tabela
    tabela_sexo.to_csv(
        os.path.join(DIR_TABELAS, "distribuicao_sexo.csv"),
        index=False,
        encoding='utf-8'
    )

In [ ]:
display(Markdown("Distribuição Geográfica"))

# REGIÃO
if 'REGIAO' in df.columns:
    dist_regiao = df['REGIAO'].value_counts()
    dist_regiao_pct = df['REGIAO'].value_counts(normalize=True) * 100
    
    tabela_regiao = pd.DataFrame({
        'Região': dist_regiao.index,
        'N': dist_regiao.values,
        'Percentual (%)': dist_regiao_pct.values
    })
    
    # Ordenar por região
    ordem_regioes = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
    tabela_regiao['Ordem'] = tabela_regiao['Região'].apply(
        lambda x: ordem_regioes.index(x) if x in ordem_regioes else 99
    )
    tabela_regiao = tabela_regiao.sort_values('Ordem').drop('Ordem', axis=1)
    
    display(Markdown("**Região de Residência**"))
    display(tabela_regiao.style.format({
        'N': '{:,.0f}',
        'Percentual (%)': '{:.2f}%'
    }).background_gradient(subset=['N'], cmap='YlGnBu'))
    
    # Salvar tabela
    tabela_regiao.to_csv(
        os.path.join(DIR_TABELAS, "distribuicao_regiao.csv"),
        index=False,
        encoding='utf-8'
    )

# UF
if 'UF_SIGLA' in df.columns:
    dist_uf = df['UF_SIGLA'].value_counts().head(15)  # Top 15 UFs
    dist_uf_pct = df['UF_SIGLA'].value_counts(normalize=True).head(15) * 100
    
    tabela_uf = pd.DataFrame({
        'UF': dist_uf.index,
        'N': dist_uf.values,
        'Percentual (%)': dist_uf_pct.values
    })
    
    display(Markdown("UF de Residência"))
    display(tabela_uf.style.format({
        'N': '{:,.0f}',
        'Percentual (%)': '{:.2f}%'
    }).background_gradient(subset=['N'], cmap='YlGnBu'))
    
    # Salvar tabela completa
    dist_uf_completo = df['UF_SIGLA'].value_counts()
    dist_uf_completo.to_csv(
        os.path.join(DIR_TABELAS, "distribuicao_uf_completo.csv"),
        encoding='utf-8'
    )

In [ ]:
display(Markdown("Características de Internação e Custo"))

# DIAS DE PERMANÊNCIA
if 'DIAS_PERM' in df.columns:
    dias_stats = {
        'N': df['DIAS_PERM'].notna().sum(),
        'Média': df['DIAS_PERM'].mean(),
        'Mediana': df['DIAS_PERM'].median(),
        'Desvio Padrão': df['DIAS_PERM'].std(),
        'Mínimo': df['DIAS_PERM'].min(),
        'Máximo': df['DIAS_PERM'].max(),
        'Q1 (25%)': df['DIAS_PERM'].quantile(0.25),
        'Q3 (75%)': df['DIAS_PERM'].quantile(0.75),
        'IIQ': df['DIAS_PERM'].quantile(0.75) - df['DIAS_PERM'].quantile(0.25)
    }
    
    tabela_dias = pd.DataFrame([dias_stats]).rename(columns={
        'N': 'N Válidos', 'IIQ': 'Intervalo Interquartil'
    })
    
    display(Markdown("**Dias de Permanência**"))
    display(tabela_dias.style.format({
        'N Válidos': '{:,.0f}',
        'Média': '{:.2f}',
        'Mediana': '{:.2f}',
        'Desvio Padrão': '{:.2f}',
        'Mínimo': '{:.1f}',
        'Máximo': '{:.1f}',
        'Q1 (25%)': '{:.1f}',
        'Q3 (75%)': '{:.1f}',
        'Intervalo Interquartil': '{:.2f}'
    }))

#CUSTO TOTAL
if 'VAL_TOT' in df.columns:
    custo_stats = {
        'N': df['VAL_TOT'].notna().sum(),
        'Média': df['VAL_TOT'].mean(),
        'Mediana': df['VAL_TOT'].median(),
        'Desvio Padrão': df['VAL_TOT'].std(),
        'Mínimo': df['VAL_TOT'].min(),
        'Máximo': df['VAL_TOT'].max(),
        'Q1 (25%)': df['VAL_TOT'].quantile(0.25),
        'Q3 (75%)': df['VAL_TOT'].quantile(0.75),
        'IIQ': df['VAL_TOT'].quantile(0.75) - df['VAL_TOT'].quantile(0.25)
    }
    
    tabela_custo = pd.DataFrame([custo_stats]).rename(columns={
        'N': 'N Válidos', 'IIQ': 'Intervalo Interquartil'
    })
    
    display(Markdown("**Custo Total (R$)**"))
    display(tabela_custo.style.format({
        'N Válidos': '{:,.0f}',
        'Média': 'R$ {:,.2f}',
        'Mediana': 'R$ {:,.2f}',
        'Desvio Padrão': 'R$ {:,.2f}',
        'Mínimo': 'R$ {:,.2f}',
        'Máximo': 'R$ {:,.2f}',
        'Q1 (25%)': 'R$ {:,.2f}',
        'Q3 (75%)': 'R$ {:,.2f}',
        'Intervalo Interquartil': 'R$ {:,.2f}'
    }))
    
    # Salvar tabela
    tabela_custo.to_csv(
        os.path.join(DIR_TABELAS, "estatisticas_custo.csv"),
        index=False,
        encoding='utf-8'
    )

In [ ]:
display(Markdown("Estatísticas Descritivas por Tipo de Procedimento"))

variaveis_strat = {
    'IDADE': 'Idade (anos)',
    'DIAS_PERM': 'Dias de Permanência',
    'VAL_TOT': 'Custo Total (R$)'
}

todas_tabelas_strat = {}

for var, nome_var in variaveis_strat.items():
    if var not in df.columns:
        continue
    
    display(Markdown(f"### {nome_var} por Procedimento"))
    
    stats_proc = []
    for proc in df['NOME_PROCEDIMENTO'].unique():
        dados = df[df['NOME_PROCEDIMENTO'] == proc][var].dropna()
        if len(dados) > 0:
            stats_proc.append({
                'Procedimento': proc,
                'N': len(dados),
                'Média': dados.mean(),
                'Mediana': dados.median(),
                'Desvio Padrão': dados.std(),
                'Mínimo': dados.min(),
                'Máximo': dados.max(),
                'Q1 (25%)': dados.quantile(0.25),
                'Q3 (75%)': dados.quantile(0.75)
            })
    
    df_stats = pd.DataFrame(stats_proc)
    todas_tabelas_strat[f'{nome_var}'] = df_stats
    
    display(df_stats.style.format({
        'N': '{:,.0f}',
        'Média': '{:.2f}' if var != 'VAL_TOT' else 'R$ {:,.2f}',
        'Mediana': '{:.2f}' if var != 'VAL_TOT' else 'R$ {:,.2f}',
        'Desvio Padrão': '{:.2f}' if var != 'VAL_TOT' else 'R$ {:,.2f}',
        'Mínimo': '{:.2f}' if var != 'VAL_TOT' else 'R$ {:,.2f}',
        'Máximo': '{:.2f}' if var != 'VAL_TOT' else 'R$ {:,.2f}',
        'Q1 (25%)': '{:.2f}' if var != 'VAL_TOT' else 'R$ {:,.2f}',
        'Q3 (75%)': '{:.2f}' if var != 'VAL_TOT' else 'R$ {:,.2f}'
    }).background_gradient(subset=['Mediana'], cmap='YlOrRd'))

# Salvar todas as tabelas estratificadas
for nome, tabela in todas_tabelas_strat.items():
    nome_arquivo = nome.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('/', '_')
    tabela.to_csv(
        os.path.join(DIR_TABELAS, f"estratificado_{nome_arquivo}.csv"),
        index=False,
        encoding='utf-8'
    )

In [ ]:
display(Markdown("Validação de Consistência Numérica"))

# Verificar consistência entre totais
validacoes = []

# Total geral
total_geral = len(df)
validacoes.append({
    'Verificação': 'Total de registros',
    'Valor Esperado': total_geral,
    'Valor Calculado': total_geral,
    'Status': 'OK' if total_geral == total_geral else 'ERRO'
})

# Soma por procedimento
total_proc = df['NOME_PROCEDIMENTO'].value_counts().sum()
validacoes.append({
    'Verificação': 'Soma por procedimento',
    'Valor Esperado': total_geral,
    'Valor Calculado': total_proc,
    'Status': 'OK' if total_proc == total_geral else 'ERRO'
})

# Soma por região
if 'REGIAO' in df.columns:
    total_regiao = df['REGIAO'].value_counts().sum()
    validacoes.append({
        'Verificação': 'Soma por região',
        'Valor Esperado': total_geral,
        'Valor Calculado': total_regiao,
        'Status': 'OK' if total_regiao == total_geral else 'ERRO'
    })

# Soma por sexo
if 'SEXO_DESC' in df.columns:
    total_sexo = df['SEXO_DESC'].value_counts().sum()
    validacoes.append({
        'Verificação': 'Soma por sexo',
        'Valor Esperado': total_geral,
        'Valor Calculado': total_sexo,
        'Status': 'OK' if total_sexo == total_geral else 'ERRO'
    })

df_validacao = pd.DataFrame(validacoes)
display(df_validacao.style.apply(
    lambda x: ['background-color: #d4edda' if v == 'OK' else 'background-color: #f8d7da' 
               for v in x],
    subset=['Status']
))

# Salvar relatório de validação
df_validacao.to_csv(
    os.path.join(DIR_DADOS, "validacao_consistencia.csv"),
    index=False,
    encoding='utf-8'
)

logger.info("Validação de consistência concluída")

In [ ]:
display(Markdown("Visualização Gráfica"))

# Violin Plot: Idade por Procedimento
plt.figure(figsize=(14, 8))

# Preparar dados
dados_violin = []
nomes_violin = []
for proc in sorted(df['NOME_PROCEDIMENTO'].unique()):
    dados = df[df['NOME_PROCEDIMENTO'] == proc]['IDADE'].dropna()
    if len(dados) > 0:
        dados_violin.append(dados)
        nomes_violin.append(proc)

# Criar violin plot
parts = plt.violinplot(dados_violin, positions=range(len(dados_violin)), 
                       showmeans=False, showmedians=True, showextrema=True,
                       widths=0.7)

# Colorir violinos
cores = sns.color_palette("Set2", len(dados_violin))
for i, (pc, cor) in enumerate(zip(parts['bodies'], cores)):
    pc.set_facecolor(cor)
    pc.set_alpha(0.6)
    pc.set_edgecolor('black')
    pc.set_linewidth(1.5)

# Personalizar medianas
for partname in ('cbars', 'cmins', 'cmaxes', 'cmedians'):
    vp = parts[partname]
    vp.set_edgecolor('black')
    vp.set_linewidth(2)

# Configurar gráfico
plt.xticks(range(len(nomes_violin)), nomes_violin, rotation=15, ha='right', fontsize=11)
plt.ylabel('Idade (anos)', fontsize=12, fontweight='bold')
plt.title('Distribuição de Idade por Tipo de Procedimento Bariátrico', 
          fontsize=14, fontweight='bold', pad=15)
plt.grid(axis='y', alpha=0.3, linestyle='--')

# Adicionar estatísticas
for i, dados in enumerate(dados_violin):
    n = len(dados)
    mediana = dados.median()
    plt.text(i, plt.ylim()[1] * 0.95, f'N={n:,}\nMd={mediana:.1f}', 
             ha='center', va='top', fontsize=9,
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', 
                      boxstyle='round,pad=0.3'))

plt.tight_layout()
plt.savefig(
    os.path.join(DIR_GRAFICOS, 'violinplot_idade_procedimento.png'),
    dpi=600,
    bbox_inches='tight'
)
plt.show()

logger.info("Violin plot de idade salvo")

In [ ]:
# Violin Plot: Custo por Procedimento
if 'VAL_TOT' in df.columns:
    plt.figure(figsize=(14, 8))
    
    # Preparar dados
    dados_violin_custo = []
    nomes_violin_custo = []
    for proc in sorted(df['NOME_PROCEDIMENTO'].unique()):
        dados = df[df['NOME_PROCEDIMENTO'] == proc]['VAL_TOT'].dropna()
        if len(dados) > 0:
            dados_violin_custo.append(dados)
            nomes_violin_custo.append(proc)
    
    # Criar violin plot
    parts = plt.violinplot(dados_violin_custo, positions=range(len(dados_violin_custo)), 
                           showmeans=False, showmedians=True, showextrema=True,
                           widths=0.7)
    
    # Colorir violinos
    cores = sns.color_palette("Set2", len(dados_violin_custo))
    for i, (pc, cor) in enumerate(zip(parts['bodies'], cores)):
        pc.set_facecolor(cor)
        pc.set_alpha(0.6)
        pc.set_edgecolor('black')
        pc.set_linewidth(1.5)
    
    # Personalizar medianas
    for partname in ('cbars', 'cmins', 'cmaxes', 'cmedians'):
        vp = parts[partname]
        vp.set_edgecolor('black')
        vp.set_linewidth(2)
    
    # Configurar gráfico
    plt.xticks(range(len(nomes_violin_custo)), nomes_violin_custo, rotation=15, ha='right', fontsize=11)
    plt.ylabel('Custo Total (R$)', fontsize=12, fontweight='bold')
    plt.title('Distribuição de Custo Total por Tipo de Procedimento Bariátrico', 
              fontsize=14, fontweight='bold', pad=15)
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Format eixo Y para moeda
    from matplotlib.ticker import FuncFormatter
    plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'R$ {x:,.0f}'))
    
    # Adicionar estatísticas
    for i, dados in enumerate(dados_violin_custo):
        n = len(dados)
        mediana = dados.median()
        plt.text(i, plt.ylim()[1] * 0.95, f'N={n:,}\nMd=R$ {mediana:,.0f}', 
                 ha='center', va='top', fontsize=9,
                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', 
                          boxstyle='round,pad=0.3'))
    
    plt.tight_layout()
    plt.savefig(
        os.path.join(DIR_GRAFICOS, 'violinplot_custo_procedimento.png'),
        dpi=600,
        bbox_inches='tight'
    )
    plt.show()
    
    logger.info("Violin plot de custo salvo")

In [ ]:
display(Markdown("Análise Descritiva Concluído"))

resumo_3 = {
    'Total de registros analisados': len(df),
    'Período': f"{df['ANO_CMPT'].min():.0f} - {df['ANO_CMPT'].max():.0f}",
    'Tipos de procedimento': df['NOME_PROCEDIMENTO'].nunique(),
    'UFs cobertas': df['UF_SIGLA'].nunique() if 'UF_SIGLA' in df.columns else 0,
    'Regiões cobertas': df['REGIAO'].nunique() if 'REGIAO' in df.columns else 0,
    'Tabelas exportadas': len(todas_tabelas_strat) + 4,
    'Gráficos exportados': 2
}

display(pd.DataFrame(list(resumo_3.items()), columns=['Parâmetro', 'Valor']).style.set_properties(**{'text-align': 'left'}))

logger.info("Análise Descritiva CONCLUÍDO COM SUCESSO")
logger.info(f"Total de registros: {len(df):,}")
logger.info(f"Tabelas salvas em: {DIR_TABELAS}")
logger.info(f"Gráficos salvas em: {DIR_GRAFICOS}")

print("\nAnálise Descritiva - CONCLUÍDO")
print(f"Tabelas: {os.path.abspath(DIR_TABELAS)}")
print(f"Gráficos: {os.path.abspath(DIR_GRAFICOS)}")

In [ ]:
# ANÁLISE DE SÉRIES TEMPORAIS

In [ ]:
#PREPARAÇÃO DA SÉRIE TEMPORAL MENSAL

# Garantir tipos numéricos
df['ANO_CMPT'] = pd.to_numeric(df['ANO_CMPT'], errors='coerce').astype('Int64')
df['MES_CMPT'] = pd.to_numeric(df['MES_CMPT'], errors='coerce').astype('Int64')

# Criar competência YYYY-MM
df['COMPETENCIA'] = df['ANO_CMPT'].astype(str) + '-' + df['MES_CMPT'].astype(str).str.zfill(2)

# Variável temporal sequencial
df_ordenado = df.sort_values(['ANO_CMPT', 'MES_CMPT']).copy()
df_ordenado['TEMPO_SEQUENCIAL'] = pd.factorize(df_ordenado['COMPETENCIA'])[0] + 1

# Dummy de pandemia (Março/2020)
df_ordenado['PANDEMIA'] = np.where(
    (df_ordenado['ANO_CMPT'] > 2020) | 
    ((df_ordenado['ANO_CMPT'] == 2020) & (df_ordenado['MES_CMPT'] >= 3)),
    1, 0
)

logger.info(f"Período: {df_ordenado['COMPETENCIA'].min()} a {df_ordenado['COMPETENCIA'].max()}")

#AGREGAÇÃO MENSAL E VARIÁVEIS DO MODELO

serie_mensal = df_ordenado.groupby(['ANO_CMPT', 'MES_CMPT', 'COMPETENCIA']).size().reset_index(name='N_PROCEDIMENTOS')
serie_mensal['TEMPO'] = np.arange(1, len(serie_mensal) + 1)
serie_mensal['PANDEMIA'] = np.where(
    (serie_mensal['ANO_CMPT'] > 2020) | 
    ((serie_mensal['ANO_CMPT'] == 2020) & (serie_mensal['MES_CMPT'] >= 3)),
    1, 0
)
serie_mensal['TEMPO_APOS_PANDEMIA'] = serie_mensal['TEMPO'] * serie_mensal['PANDEMIA']

display(Markdown(f"**Período:** {serie_mensal['COMPETENCIA'].min()} a {serie_mensal['COMPETENCIA'].max()}"))
display(Markdown(f"**Total de meses:** {len(serie_mensal)} | Pré-pandemia: {(serie_mensal['PANDEMIA']==0).sum()} | Pós-pandemia: {(serie_mensal['PANDEMIA']==1).sum()}"))

In [ ]:
# VISUALIZAÇÃO DA SÉRIE TEMPORAL

plt.figure(figsize=(16, 8))
plt.plot(serie_mensal['TEMPO'], serie_mensal['N_PROCEDIMENTOS'], 
         linewidth=2.5, color='#2c3e50', label='Procedimentos Mensais', marker='o', markersize=4)

serie_mensal['MEDIA_MOVEL_6M'] = serie_mensal['N_PROCEDIMENTOS'].rolling(window=6, center=True, min_periods=1).mean()
plt.plot(serie_mensal['TEMPO'], serie_mensal['MEDIA_MOVEL_6M'], 
         linewidth=3.0, color='#e74c3c', linestyle='--', label='Média Móvel (6 meses)')

tempo_pandemia = serie_mensal[serie_mensal['PANDEMIA'] == 1]['TEMPO'].min()
plt.axvline(x=tempo_pandemia, color='#27ae60', linewidth=2.5, linestyle='-', label='Início da Pandemia (Mar/2020)', alpha=0.8)

plt.xlabel('Meses da Série Temporal', fontsize=12, fontweight='bold')
plt.ylabel('Número de Procedimentos', fontsize=12, fontweight='bold')
plt.title('Evolução Temporal das Cirurgias Bariátricas no SUS (2016-2025)', fontsize=14, fontweight='bold', pad=15)
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3, linestyle='--')

tick_positions = serie_mensal.loc[serie_mensal['MES_CMPT'] == 1, 'TEMPO'].values
tick_labels = serie_mensal.loc[serie_mensal['MES_CMPT'] == 1, 'ANO_CMPT'].astype(str).values
plt.xticks(tick_positions, tick_labels, rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, 'serie_temporal_bruta.png'), dpi=600, bbox_inches='tight')
plt.show()
logger.info("Gráfico de série temporal salvo")

In [ ]:
# TESTE 1: ESTACIONARIEDADE (ADF)
display(Markdown("## TESTE 1: Estacionariedade (Dickey-Fuller Aumentado)"))

resultado_adf = adfuller(serie_mensal['N_PROCEDIMENTOS'], autolag='AIC')
adf_stat, adf_pvalor, adf_criticos = resultado_adf[0], resultado_adf[1], resultado_adf[4]

tabela_adf = pd.DataFrame({
    'Estatística ADF': [adf_stat],
    'P-valor': [adf_pvalor],
    'Decisão (α=0,05)': ['Não-estacionária' if adf_pvalor > 0.05 else 'Estacionária']
    })

display(tabela_adf.style.format({'Estatística ADF': '{:.4f}', 'P-valor': '{:.4f}'}))
display(Markdown("**Valores Críticos:**"))
for nivel, valor in adf_criticos.items():
    display(Markdown(f"- {nivel}: {valor:.4f}"))

if adf_pvalor > 0.05:
    display(Markdown("**Série NÃO-ESTACIONÁRIA:** Diferenciação aplicada"))
    serie_mensal['N_PROCEDIMENTOS_DIFF'] = serie_mensal['N_PROCEDIMENTOS'].diff().dropna()
    serie_mensal = serie_mensal.dropna(subset=['N_PROCEDIMENTOS_DIFF']).reset_index(drop=True)
else:
    display(Markdown("**Série ESTACIONÁRIA:** Prosseguir para autocorrelação"))

pd.DataFrame([{'estatistica': adf_stat, 'p_valor': adf_pvalor, 'estacionaria': adf_pvalor <= 0.05}]).to_csv(
    os.path.join(DIR_DADOS, "teste_adf.csv"), index=False, encoding='utf-8')

In [ ]:
#TESTE 2: AUTOCORRELAÇÃO (DURBIN-WATSON)

display(Markdown("## TESTE 2: Autocorrelação (Durbin-Watson)"))

X_ols = sm.add_constant(serie_mensal['TEMPO'])
y_ols = serie_mensal['N_PROCEDIMENTOS']
modelo_ols = sm.OLS(y_ols, X_ols).fit()
dw_stat = durbin_watson(modelo_ols.resid)

display(Markdown(f"**Durbin-Watson:** {dw_stat:.4f}"))

if dw_stat < 1.5:
    interpretacao_dw, correcao_necessaria = "Autocorrelação positiva", "Correção recomendada"
elif dw_stat > 2.5:
    interpretacao_dw, correcao_necessaria = "Autocorrelação negativa", "Correção recomendada"
else:
    interpretacao_dw, correcao_necessaria = "Sem autocorrelação significativa", "OLS adequado"

display(Markdown(f"**Interpretação:** {interpretacao_dw} | **Decisão:** {correcao_necessaria}"))

# Gráficos ACF/PACF
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
acf_vals, acf_confint = acf(modelo_ols.resid, nlags=24, alpha=0.05)
ax1.stem(range(len(acf_vals)), acf_vals, linefmt='b-', markerfmt='bo', basefmt='k-')
ax1.axhline(y=acf_confint[0][0], color='r', linestyle='--', alpha=0.5)
ax1.axhline(y=acf_confint[0][1], color='r', linestyle='--', alpha=0.5)
ax1.set_title('ACF dos Resíduos', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

pacf_vals, pacf_confint = pacf(modelo_ols.resid, nlags=24, alpha=0.05)
ax2.stem(range(len(pacf_vals)), pacf_vals, linefmt='g-', markerfmt='go', basefmt='k-')
ax2.axhline(y=pacf_confint[0][0], color='r', linestyle='--', alpha=0.5)
ax2.axhline(y=pacf_confint[0][1], color='r', linestyle='--', alpha=0.5)
ax2.set_title('PACF dos Resíduos', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, 'acf_pacf_residuos.png'), dpi=600, bbox_inches='tight')
plt.show()

pd.DataFrame([{'estatistica_dw': dw_stat, 'interpretacao': interpretacao_dw, 'correcao': correcao_necessaria}]).to_csv(
    os.path.join(DIR_DADOS, "teste_durbin_watson.csv"), index=False, encoding='utf-8')
logger.info(f"Durbin-Watson: {dw_stat:.4f} - {interpretacao_dw}")

In [ ]:
# TESTE 3: MODELO DE REGRESSÃO SEGMENTADA (ITS) COM ERROS-PADRÃO ROBUSTOS (HAC)

display(Markdown("Impacto da Pandemia (Interrupted Time Series com Erros-Padrão HAC)"))

# Preparar variáveis
X_its = serie_mensal[['TEMPO', 'PANDEMIA', 'TEMPO_APOS_PANDEMIA']].copy()
X_its = sm.add_constant(X_its)
y_its = serie_mensal['N_PROCEDIMENTOS']

# Ajustar OLS com erros-padrão Newey-West (HAC)
modelo_hac = sm.OLS(y_its, X_its).fit(cov_type='HAC', cov_kwds={'maxlags': 12})

# Extrair resultados
coeficientes = modelo_hac.params
erros = modelo_hac.bse
p_valores = modelo_hac.pvalues 
ic95 = modelo_hac.conf_int()
dw_hac = durbin_watson(modelo_hac.resid)

# Função para formatar p-valor 
def formatar_pvalor(p):
    if pd.isna(p):
        return 'p-valor=NA'
    elif p < 0.001:
        return 'p-valor<0,001'
    else:
        return f'p-valor={p:.3f}'.replace('.', ',')

# Criar tabela com valores
tabela_its = pd.DataFrame({
    'Parâmetro': ['Intercepto (β₀)', 'Tendência Pré (β₁)', 'Mudança Nível (β₂)', 'Mudança Inclinação (β₃)'],
    'Coeficiente': coeficientes.values,  
    'Erro Padrão': erros.values,         
    'IC95% Inferior': ic95.iloc[:, 0].values,  
    'IC95% Superior': ic95.iloc[:, 1].values,  
    'P_valor_bruto': p_valores.values    
})

# Formatar para exibição 
display(Markdown("### Coeficientes do Modelo ITS (Erros-Padrão HAC)"))

# Aplicar formatação apenas nas colunas numéricas
display(tabela_its.style.format({
    'Coeficiente': '{:.2f}',
    'Erro Padrão': '{:.2f}',
    'IC95% Inferior': '{:.2f}',
    'IC95% Superior': '{:.2f}'
}).set_properties(**{'text-align': 'center'}))

# Exibir p-valores formatados
p_valores_formatados = tabela_its['P_valor_bruto'].apply(formatar_pvalor)
display(Markdown(f"**P-valores:**"))
for param, p_formatado in zip(tabela_its['Parâmetro'], p_valores_formatados):
    display(Markdown(f"- {param}: {p_formatado}"))

# Interpretação
if p_valores['PANDEMIA'] < 0.05:
    display(Markdown(f"**Impacto da pandemia significativo** ({formatar_pvalor(p_valores['PANDEMIA'])})"))
else:
    display(Markdown(f"**Impacto da pandemia não significativo** ({formatar_pvalor(p_valores['PANDEMIA'])})"))

if p_valores['TEMPO_APOS_PANDEMIA'] < 0.05:
    display(Markdown(f"**Mudança na tendência pós-pandemia significativa** ({formatar_pvalor(p_valores['TEMPO_APOS_PANDEMIA'])})"))

# Salvar tabela
tabela_export = tabela_its.copy()
tabela_export['P-valor (RESS)'] = tabela_export['P_valor_bruto'].apply(formatar_pvalor)
tabela_export = tabela_export.drop('P_valor_bruto', axis=1)  

tabela_export.to_csv(
    os.path.join(DIR_TABELAS, "modelo_regressao_segmentada.csv"),
    index=False,
    encoding='utf-8',
    sep=';'
)

logger.info(f"Modelo ITS ajustado (OLS+HAC) | DW={dw_hac:.2f} | Lags HAC=12")

In [ ]:

# VISUALIZAÇÃO DO MODELO ITS COM PARÂMETROS DA EQUAÇÃO (HAC)


display(Markdown("### Gráfico do Modelo de Regressão Segmentada (Erros-Padrão HAC)"))

serie_mensal['PREDITO_ITS'] = modelo_hac.fittedvalues
tempo_transicao = serie_mensal[serie_mensal['PANDEMIA'] == 1]['TEMPO'].min()

plt.figure(figsize=(16, 9))

# Dados observados
plt.scatter(serie_mensal['TEMPO'], serie_mensal['N_PROCEDIMENTOS'], 
            alpha=0.6, s=40, color='#34495e', label='Observado', zorder=5)

# Tendência pré-pandemia
pre = serie_mensal[serie_mensal['PANDEMIA'] == 0].copy()
pre_pred = coeficientes['const'] + coeficientes['TEMPO'] * pre['TEMPO']
plt.plot(pre['TEMPO'], pre_pred, linewidth=3.0, color='#27ae60', 
         linestyle='-', label='Tendência Pré-Pandemia')

# Tendência pós-pandemia
pos = serie_mensal[serie_mensal['PANDEMIA'] == 1].copy()
pos_pred = (coeficientes['const'] + coeficientes['TEMPO'] * pos['TEMPO'] + 
            coeficientes['PANDEMIA'] + coeficientes['TEMPO_APOS_PANDEMIA'] * pos['TEMPO_APOS_PANDEMIA'])
plt.plot(pos['TEMPO'], pos_pred, linewidth=3.0, color='#e74c3c', 
         linestyle='-', label='Tendência Pós-Pandemia')

# Linha da pandemia
plt.axvline(x=tempo_transicao, color='#95a5a6', linewidth=2.5, linestyle='--', 
            label='Início da Pandemia (Mar/2020)', alpha=0.8)

# CORREÇÃO: Calcular R² para modelo HAC
r_squared = modelo_hac.rsquared

equacao_texto = (
    f"Equação do Modelo (OLS + HAC):\n"
    f"Yₜ = β₀ + β₁·TEMPO + β₂·PANDEMIA + β₃·TEMPOpós + εₜ\n\n"
    f"β₀ (Intercepto):           {coeficientes['const']:.2f}\n"
    f"β₁ (Tendência Pré):        {coeficientes['TEMPO']:+.2f} proc/mês\n"
    f"β₂ (Mudança Nível):        {coeficientes['PANDEMIA']:+.2f} proc\n"
    f"β₃ (Mudança Inclinação):   {coeficientes['TEMPO_APOS_PANDEMIA']:+.2f} proc/mês\n\n"
    f"R² = {r_squared:.3f} | Erros-padrão: Newey-West (12 lags)"
)

plt.text(0.02, 0.98, equacao_texto, transform=plt.gca().transAxes,
         fontsize=9, fontfamily='monospace', verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.4, edgecolor='gray'))

plt.xlabel('Meses da Série Temporal', fontsize=12, fontweight='bold')
plt.ylabel('Número de Procedimentos', fontsize=12, fontweight='bold')
plt.title('Modelo de Regressão Segmentada: Impacto da Pandemia (Erros-Padrão HAC)', 
          fontsize=14, fontweight='bold', pad=15)
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3, linestyle='--')

tick_pos = serie_mensal.loc[serie_mensal['MES_CMPT'] == 1, 'TEMPO'].values
tick_lbl = serie_mensal.loc[serie_mensal['MES_CMPT'] == 1, 'ANO_CMPT'].astype(str).values
plt.xticks(tick_pos, tick_lbl, rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, 'modelo_regressao_segmentada_parametros.png'), 
            dpi=600, bbox_inches='tight')
plt.show()

logger.info("Gráfico do modelo ITS com parâmetros salvo (HAC)")

In [ ]:
# SEGUNDA VISUALIZAÇÃO


display(Markdown("### Gráfico do Modelo de Regressão Segmentada"))

serie_mensal['PREDITO_ITS'] = modelo_hac.fittedvalues
tempo_transicao = serie_mensal[serie_mensal['PANDEMIA'] == 1]['TEMPO'].min()

plt.figure(figsize=(16, 9))

plt.scatter(serie_mensal['TEMPO'], serie_mensal['N_PROCEDIMENTOS'], 
            alpha=0.6, s=40, color='#34495e', label='Observado', zorder=5)

pre = serie_mensal[serie_mensal['PANDEMIA'] == 0].copy()
pre_pred = coeficientes['const'] + coeficientes['TEMPO'] * pre['TEMPO']
plt.plot(pre['TEMPO'], pre_pred, linewidth=3.0, color='#27ae60', 
         linestyle='-', label='Tendência Pré-Pandemia')

pos = serie_mensal[serie_mensal['PANDEMIA'] == 1].copy()
pos_pred = (coeficientes['const'] + coeficientes['TEMPO'] * pos['TEMPO'] + 
            coeficientes['PANDEMIA'] + coeficientes['TEMPO_APOS_PANDEMIA'] * pos['TEMPO_APOS_PANDEMIA'])
plt.plot(pos['TEMPO'], pos_pred, linewidth=3.0, color='#e74c3c', 
         linestyle='-', label='Tendência Pós-Pandemia')

plt.axvline(x=tempo_transicao, color='#95a5a6', linewidth=2.5, linestyle='--', 
            label='Início da Pandemia (Mar/2020)', alpha=0.8)

plt.xlabel('Meses da Série Temporal', fontsize=12, fontweight='bold')
plt.ylabel('Número de Procedimentos', fontsize=12, fontweight='bold')
plt.title('Modelo de Regressão Segmentada: Impacto da Pandemia', 
          fontsize=14, fontweight='bold', pad=15)
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3, linestyle='--')

tick_pos = serie_mensal.loc[serie_mensal['MES_CMPT'] == 1, 'TEMPO'].values
tick_lbl = serie_mensal.loc[serie_mensal['MES_CMPT'] == 1, 'ANO_CMPT'].astype(str).values
plt.xticks(tick_pos, tick_lbl, rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, 'modelo_regressao_segmentada.png'), 
            dpi=600, bbox_inches='tight')
plt.show()

logger.info("Gráfico do modelo ITS salvo")

In [ ]:
# RESUMO E VALIDAÇÃO (CORRIGIDO)


display(Markdown("Análise de Séries Temporais Concluída"))

resumo = {
    'Teste ADF': f"P-valor = {adf_pvalor:.4f} - {'Estacionária' if adf_pvalor <= 0.05 else 'Não-estacionária'}",
    'Durbin-Watson (pré-correção)': f"DW = {dw_stat:.4f} - {interpretacao_dw}",
    'Modelo': 'OLS com Erros-Padrão HAC (Newey-West)',
    'Impacto Pandemia (β₂)': f"{coeficientes['PANDEMIA']:.2f} ({formatar_pvalor(p_valores['PANDEMIA'])})",
    'Mudança Tendência (β₃)': f"{coeficientes['TEMPO_APOS_PANDEMIA']:.2f} proc/mês ({formatar_pvalor(p_valores['TEMPO_APOS_PANDEMIA'])})",
    'Período': f"{serie_mensal['COMPETENCIA'].min()} a {serie_mensal['COMPETENCIA'].max()}",
    'Total de Meses': len(serie_mensal)
}

display(pd.DataFrame(list(resumo.items()), columns=['Métrica', 'Resultado']).style.set_properties(**{'text-align': 'left'}))

logger.info("Análise de Séries Temporais Concluída")
logger.info(f"ADF: {adf_pvalor:.4f} | DW pré-correção: {dw_stat:.4f} | Modelo: OLS+HAC")
logger.info(f"Impacto pandemia: β₂={coeficientes['PANDEMIA']:.2f} (p-valor={p_valores['PANDEMIA']:.4f})")
logger.info(f"Mudança tendência: β₃={coeficientes['TEMPO_APOS_PANDEMIA']:.2f} (p-valor={p_valores['TEMPO_APOS_PANDEMIA']:.4f})")
logger.info("="*60)

print(f"\n Análise concluída | Tabelas: {DIR_TABELAS} | Gráficos: {DIR_GRAFICOS}")

In [ ]:

# ANÁLISE TEMPORAL ESTRATIFICADA POR PROCEDIMENTO

display(Markdown("## Análise Temporal Estratificada por Procedimento"))

resultados_temporais = []

for cod_proc, nome_proc in CODIGOS_BARIATRICOS.items():
    df_proc = df[df['PROC_REA'] == cod_proc].copy()
    if len(df_proc) == 0:
        continue
    
    # Agrupar por mês
    serie_mensal = df_proc.groupby(['ANO_CMPT', 'MES_CMPT']).size().reset_index(name='N_PROC')
    serie_mensal['DATA'] = pd.to_datetime(
        serie_mensal['ANO_CMPT'].astype(str) + '-' + 
        serie_mensal['MES_CMPT'].astype(str).str.zfill(2) + '-01'
    )
    serie_mensal = serie_mensal.sort_values('DATA')
    
    if len(serie_mensal) < 12:
        continue
    
    # Variável temporal sequencial
    serie_mensal['TEMPO'] = np.arange(1, len(serie_mensal) + 1)
    
    # Tendência linear
    from scipy.stats import linregress
    slope, intercept, r_value, p_value, std_err = linregress(
        serie_mensal['TEMPO'], serie_mensal['N_PROC']
    )
    taxa_crescimento = ((slope / serie_mensal['N_PROC'].mean()) * 100) if serie_mensal['N_PROC'].mean() > 0 else 0
    
    # Dummy pandemia
    serie_mensal['PANDEMIA'] = np.where(
        (serie_mensal['ANO_CMPT'] > 2020) | 
        ((serie_mensal['ANO_CMPT'] == 2020) & (serie_mensal['MES_CMPT'] >= 3)),
        1, 0
    )
    
    # Modelo simples pré/pós
    import statsmodels.api as sm
    modelo = sm.OLS(serie_mensal['N_PROC'], 
                   sm.add_constant(serie_mensal[['TEMPO', 'PANDEMIA']])).fit()
    
    resultados_temporais.append({
        'Procedimento': NOMES_CURTOS.get(cod_proc, nome_proc),
        'Codigo': cod_proc,
        'Total_Procedimentos': serie_mensal['N_PROC'].sum(),
        'Meses_Analisados': len(serie_mensal),
        'Taxa_Crescimento_Mensal_Pct': taxa_crescimento,
        'R2': r_value**2,
        'P_valor_Tendencia': p_value,
        'Impacto_Pandemia_Coef': modelo.params['PANDEMIA'],
        'P_valor_Pandemia': modelo.pvalues['PANDEMIA']
    })
    
    # Gráfico individual por procedimento
    plt.figure(figsize=(14, 8), dpi=600)
    plt.plot(serie_mensal['DATA'], serie_mensal['N_PROC'], 
            marker='o', markersize=4, linewidth=2, 
            color=CORES_PROCEDIMENTOS.get(cod_proc, '#3498db'), 
            label='Procedimentos Mensais', alpha=0.85)
    
    serie_mensal['MM_6M'] = serie_mensal['N_PROC'].rolling(6, center=True, min_periods=1).mean()
    plt.plot(serie_mensal['DATA'], serie_mensal['MM_6M'], 
            linewidth=3, color='black', linestyle='--', label='Média Móvel (6 meses)', alpha=0.7)
    
    pred_tendencia = intercept + slope * serie_mensal['TEMPO']
    plt.plot(serie_mensal['DATA'], pred_tendencia, 
            linewidth=2, color='red', linestyle=':', label=f'Tendência (R²={r_value**2:.3f})')
    
    pandemia_inicio = serie_mensal[serie_mensal['PANDEMIA'] == 1]['DATA'].min()
    if pd.notna(pandemia_inicio):
        plt.axvline(x=pandemia_inicio, color='gray', linestyle='-.', 
                   linewidth=2, label='Início Pandemia (Mar/2020)', alpha=0.6)
    
    plt.xlabel('Data', fontsize=12, fontweight='bold')
    plt.ylabel('Número de Procedimentos', fontsize=12, fontweight='bold')
    plt.title(f'Evolução Temporal: {NOMES_CURTOS.get(cod_proc, nome_proc)}\n'
             f'Taxa: {taxa_crescimento:+.2f}% ao mês | Impacto pandemia: {modelo.params["PANDEMIA"]:+.1f}',
             fontsize=14, fontweight='bold', pad=15)
    plt.legend(loc='upper left', fontsize=10)
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b/%Y'))
    plt.gca().xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
    plt.gcf().autofmt_xdate()
    plt.tight_layout()
    
    nome_arq = f"temporal_{NOMES_CURTOS.get(cod_proc, cod_proc).replace(' ', '_')}.png"
    plt.savefig(os.path.join(DIR_GRAFICOS, nome_arq), dpi=600, bbox_inches='tight')
    plt.close()

# Tabela consolidada
df_resultados_temp = pd.DataFrame(resultados_temporais)
if not df_resultados_temp.empty:
    df_resultados_temp.to_csv(
        os.path.join(DIR_TABELAS, "analise_temporal_por_procedimento.csv"),
        index=False, encoding='utf-8', sep=';'
    )
    display(Markdown("### Resumo das Tendências Temporais por Procedimento"))
    display(df_resultados_temp.style.format({
        'Total_Procedimentos': '{:,.0f}',
        'Taxa_Crescimento_Mensal_Pct': '{:+.2f}%',
        'R2': '{:.3f}',
        'P_valor_Tendencia': '{:.4f}',
        'Impacto_Pandemia_Coef': '{:+.2f}',
        'P_valor_Pandemia': '{:.4f}'
    }))

logger.info(f"Análise temporal estratificada concluída: {len(resultados_temporais)} procedimentos")

In [ ]:

# ANÁLISE TEMPORAL ESTRATIFICADA POR PROCEDIMENTO

display(Markdown("## Evolução Temporal Estratificada por Procedimento"))

# Garantir que DATA existe
if 'DATA' not in df.columns:
    df['DATA'] = pd.to_datetime(
        df['ANO_CMPT'].astype(str) + '-' + 
        df['MES_CMPT'].astype(str).str.zfill(2) + '-01',
        errors='coerce'
    )

df_temp = df.dropna(subset=['DATA']).copy()

# Criar diretório para gráficos temporais
os.makedirs(os.path.join(DIR_GRAFICOS, "temporal_por_procedimento"), exist_ok=True)

# GRÁFICOS INDIVIDUAIS POR PROCEDIMENTO

for cod_proc, nome_proc in CODIGOS_BARIATRICOS.items():
    # Filtrar dados do procedimento
    df_proc = df_temp[df_temp['PROC_REA'] == cod_proc].copy()
    if len(df_proc) == 0:
        continue
    
    # Agrupar por mês
    serie_mensal = df_proc.groupby(['ANO_CMPT', 'MES_CMPT']).size().reset_index(name='N_PROC')
    serie_mensal['DATA'] = pd.to_datetime(
        serie_mensal['ANO_CMPT'].astype(str) + '-' + 
        serie_mensal['MES_CMPT'].astype(str).str.zfill(2) + '-01'
    )
    serie_mensal = serie_mensal.sort_values('DATA').reset_index(drop=True)
    
    if len(serie_mensal) < 6:
        continue
    
    # Variáveis para modelagem
    serie_mensal['TEMPO'] = np.arange(1, len(serie_mensal) + 1)
    serie_mensal['PANDEMIA'] = np.where(
        (serie_mensal['ANO_CMPT'] > 2020) | 
        ((serie_mensal['ANO_CMPT'] == 2020) & (serie_mensal['MES_CMPT'] >= 3)),
        1, 0
    )
    
    # Média móvel (6 meses) para suavização
    serie_mensal['MM_6M'] = serie_mensal['N_PROC'].rolling(6, center=True, min_periods=1).mean()
    
    # Linha de tendência linear
    from scipy.stats import linregress
    slope, intercept, r_value, _, _ = linregress(serie_mensal['TEMPO'], serie_mensal['N_PROC'])
    serie_mensal['TENDENCIA'] = intercept + slope * serie_mensal['TEMPO']
    
    # GRÁFICO INDIVIDUAL
    plt.figure(figsize=(14, 8), dpi=600)
    
    # Série observada
    plt.plot(serie_mensal['DATA'], serie_mensal['N_PROC'], 
            marker='o', markersize=4, linewidth=2, 
            color=CORES_PROCEDIMENTOS.get(cod_proc, '#3498db'), 
            label='Procedimentos Mensais', alpha=0.85, zorder=3)
    
    # Média móvel
    plt.plot(serie_mensal['DATA'], serie_mensal['MM_6M'], 
            linewidth=3, color='black', linestyle='--', 
            label='Média Móvel (6 meses)', alpha=0.7, zorder=2)
    
    # Linha de tendência
    plt.plot(serie_mensal['DATA'], serie_mensal['TENDENCIA'], 
            linewidth=2, color='red', linestyle=':', 
            label=f'Tendência (R²={r_value**2:.3f})', zorder=1)
    
    # Linha da pandemia
    pandemia_inicio = serie_mensal[serie_mensal['PANDEMIA'] == 1]['DATA'].min()
    if pd.notna(pandemia_inicio):
        plt.axvline(x=pandemia_inicio, color='gray', linestyle='-.', 
                   linewidth=2, label='Início Pandemia (Mar/2020)', alpha=0.6, zorder=4)
    
    # Configurar gráfico
    nome_curto = NOMES_CURTOS.get(cod_proc, nome_proc)
    total_proc = serie_mensal['N_PROC'].sum()
    taxa_crescimento = ((slope / serie_mensal['N_PROC'].mean()) * 100) if serie_mensal['N_PROC'].mean() > 0 else 0
    
    plt.xlabel('Data', fontsize=12, fontweight='bold')
    plt.ylabel('Número de Procedimentos', fontsize=12, fontweight='bold')
    plt.title(f'{nome_curto}\nTotal: {total_proc:,} | Crescimento: {taxa_crescimento:+.2f}% ao mês | R²={r_value**2:.3f}', 
             fontsize=14, fontweight='bold', pad=15)
    plt.legend(loc='upper left', fontsize=10)
    plt.grid(True, alpha=0.3, linestyle='--')
    
    # Formatar eixo X
    plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b/%Y'))
    plt.gca().xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
    plt.gcf().autofmt_xdate()
    
    # Salvar gráfico individual
    nome_arq = f"temporal_{nome_curto.replace(' ', '_').lower()}.png"
    plt.savefig(os.path.join(DIR_GRAFICOS, "temporal_por_procedimento", nome_arq), 
                dpi=600, bbox_inches='tight')
    plt.close()

display(Markdown(f"**Gráficos individuais salvos em:** `{os.path.join(DIR_GRAFICOS, 'temporal_por_procedimento')}`"))

# GRÁFICO COMBINADO: SUBPLOTS POR PROCEDIMENTO

display(Markdown("### Comparação Visual: Evolução Temporal por Procedimento"))

# Criar figura com subplots
fig, axes = plt.subplots(2, 2, figsize=(18, 14), dpi=600)
axes = axes.flatten()

for idx, (cod_proc, nome_proc) in enumerate(CODIGOS_BARIATRICOS.items()):
    ax = axes[idx]
    
    # Filtrar e preparar dados
    df_proc = df_temp[df_temp['PROC_REA'] == cod_proc].copy()
    if len(df_proc) == 0:
        ax.text(0.5, 0.5, 'Sem dados', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(NOMES_CURTOS.get(cod_proc, nome_proc), fontsize=12, fontweight='bold')
        continue
    
    serie_mensal = df_proc.groupby(['ANO_CMPT', 'MES_CMPT']).size().reset_index(name='N_PROC')
    serie_mensal['DATA'] = pd.to_datetime(
        serie_mensal['ANO_CMPT'].astype(str) + '-' + 
        serie_mensal['MES_CMPT'].astype(str).str.zfill(2) + '-01'
    )
    serie_mensal = serie_mensal.sort_values('DATA')
    
    if len(serie_mensal) < 6:
        ax.text(0.5, 0.5, 'Dados insuficientes', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(NOMES_CURTOS.get(cod_proc, nome_proc), fontsize=12, fontweight='bold')
        continue
    
    # Plotar
    ax.plot(serie_mensal['DATA'], serie_mensal['N_PROC'], 
           marker='o', markersize=3, linewidth=1.5, 
           color=CORES_PROCEDIMENTOS.get(cod_proc, '#3498db'), 
           alpha=0.8, label='Observado')
    
    # Média móvel
    mm = serie_mensal['N_PROC'].rolling(6, center=True, min_periods=1).mean()
    ax.plot(serie_mensal['DATA'], mm, linewidth=2, color='black', linestyle='--', alpha=0.6, label='MM 6M')
    
    # Linha da pandemia
    pandemia = serie_mensal[(serie_mensal['ANO_CMPT'] == 2020) & (serie_mensal['MES_CMPT'] == 3)]
    if not pandemia.empty:
        ax.axvline(x=pandemia['DATA'].values[0], color='gray', linestyle='-.', 
                  linewidth=1.5, alpha=0.5)
    
    # Configurar subplot
    nome_curto = NOMES_CURTOS.get(cod_proc, nome_proc)
    total = serie_mensal['N_PROC'].sum()
    ax.set_title(f'{nome_curto} (n={total:,})', fontsize=11, fontweight='bold')
    ax.set_xlabel('Data', fontsize=9)
    ax.set_ylabel('Nº Procedimentos', fontsize=9)
    ax.tick_params(axis='both', labelsize=8)
    ax.grid(True, alpha=0.2, linestyle='--')
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
    ax.tick_params(axis='x', rotation=45)
    
    # Legenda apenas no primeiro subplot
    if idx == 0:
        ax.legend(fontsize=8, loc='upper left')

# Título geral e ajuste
fig.suptitle('Evolução Temporal de Procedimentos Bariátricos no SUS (2016-2025) por Tipo de Procedimento', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

# Salvar gráfico combinado
plt.savefig(os.path.join(DIR_GRAFICOS, 'temporal_combinado_4procedimentos.png'), 
            dpi=600, bbox_inches='tight')
plt.show()

display(Markdown(f"**Gráfico combinado salvo:** `temporal_combinado_4procedimentos.png`"))

# TABELA RESUMO DAS TENDÊNCIAS TEMPORAIS

display(Markdown("### Resumo Quantitativo das Tendências Temporais"))

resumo_tendencias = []

for cod_proc, nome_proc in CODIGOS_BARIATRICOS.items():
    df_proc = df_temp[df_temp['PROC_REA'] == cod_proc].copy()
    if len(df_proc) < 12:
        continue
    
    serie = df_proc.groupby(['ANO_CMPT', 'MES_CMPT']).size().reset_index(name='N_PROC')
    serie['TEMPO'] = np.arange(1, len(serie) + 1)
    
    slope, intercept, r_value, p_value, std_err = linregress(serie['TEMPO'], serie['N_PROC'])
    taxa_crescimento = ((slope / serie['N_PROC'].mean()) * 100) if serie['N_PROC'].mean() > 0 else 0
    
    # Impacto pandemia (comparação média pré vs pós)
    pre = serie[(serie['ANO_CMPT'] < 2020) | ((serie['ANO_CMPT'] == 2020) & (serie['MES_CMPT'] < 3))]
    pos = serie[(serie['ANO_CMPT'] > 2020) | ((serie['ANO_CMPT'] == 2020) & (serie['MES_CMPT'] >= 3))]
    
    impacto_pandemia = ((pos['N_PROC'].mean() - pre['N_PROC'].mean()) / pre['N_PROC'].mean() * 100) if len(pre) > 0 and pre['N_PROC'].mean() > 0 else np.nan
    
    resumo_tendencias.append({
        'Procedimento': NOMES_CURTOS.get(cod_proc, nome_proc),
        'Total_Procedimentos': serie['N_PROC'].sum(),
        'Meses_Analisados': len(serie),
        'Taxa_Crescimento_Mensal (%)': taxa_crescimento,
        'R²': r_value**2,
        'P_valor_Tendencia': p_value,
        'Impacto_Pandemia (%)': impacto_pandemia
    })

df_resumo = pd.DataFrame(resumo_tendencias)

display(df_resumo.style.format({
    'Total_Procedimentos': '{:,.0f}',
    'Meses_Analisados': '{:.0f}',
    'Taxa_Crescimento_Mensal (%)': '{:+.2f}%',
    'R²': '{:.3f}',
    'P_valor_Tendencia': '{:.4f}',
    'Impacto_Pandemia (%)': '{:+.1f}%'
}).background_gradient(subset=['Taxa_Crescimento_Mensal (%)'], cmap='RdYlGn'))

# Salvar tabela resumo
df_resumo.to_csv(
    os.path.join(DIR_TABELAS, "resumo_tendencias_temporais_por_procedimento.csv"),
    index=False, encoding='utf-8', sep=';'
)

logger.info("Análise temporal estratificada por procedimento concluída")
print(f"Gráficos individuais: {os.path.join(DIR_GRAFICOS, 'temporal_por_procedimento')}")
print(f"Gráfico combinado: {os.path.join(DIR_GRAFICOS, 'temporal_combinado_4procedimentos.png')}")
print(f"Tabela resumo: {os.path.join(DIR_TABELAS, 'resumo_tendencias_temporais_por_procedimento.csv')}")

In [ ]:
# EVOLUÇÃO TEMPORAL COM TODOS OS PROCEDIMENTOS SOBREPOSTOS

display(Markdown("### Gráfico Comparativo: Evolução Temporal"))

plt.figure(figsize=(16, 10), dpi=600)

# Plotar todos os procedimentos no mesmo gráfico
for cod_proc, nome_proc in CODIGOS_BARIATRICOS.items():
    df_proc = df_temp[df_temp['PROC_REA'] == cod_proc].copy()
    if len(df_proc) == 0:
        continue
    
    # Agrupar por mês
    serie_mensal = df_proc.groupby(['ANO_CMPT', 'MES_CMPT']).size().reset_index(name='N_PROC')
    serie_mensal['DATA'] = pd.to_datetime(
        serie_mensal['ANO_CMPT'].astype(str) + '-' + 
        serie_mensal['MES_CMPT'].astype(str).str.zfill(2) + '-01'
    )
    serie_mensal = serie_mensal.sort_values('DATA')
    
    if len(serie_mensal) < 6:
        continue
    
    # Plotar série observada
    plt.plot(serie_mensal['DATA'], serie_mensal['N_PROC'], 
            marker='o', markersize=5, linewidth=2.5, 
            color=CORES_PROCEDIMENTOS.get(cod_proc, '#3498db'), 
            label=NOMES_CURTOS.get(cod_proc, nome_proc), 
            alpha=0.85)

# Linha vertical da pandemia
pandemia_data = df_temp[(df_temp['ANO_CMPT'] == 2020) & (df_temp['MES_CMPT'] == 3)]['DATA'].min()
if pd.notna(pandemia_data):
    plt.axvline(x=pandemia_data, color='gray', linestyle='--', 
               linewidth=2.5, label='Início Pandemia (Mar/2020)', alpha=0.7)

# Configurar gráfico
plt.xlabel('Data', fontsize=13, fontweight='bold')
plt.ylabel('Número de Procedimentos', fontsize=13, fontweight='bold')
plt.title('Evolução Temporal Comparativa: Procedimentos Bariátricos no SUS (2016-2025)', 
         fontsize=16, fontweight='bold', pad=20)
plt.legend(title='Tipo de Procedimento', title_fontsize=12, fontsize=11, loc='upper left')
plt.grid(True, alpha=0.3, linestyle='--')

# Formatar eixo X
plt.gca().xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b/%Y'))
plt.gca().xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
plt.gcf().autofmt_xdate()

# Ajustar layout e salvar
plt.tight_layout()
plt.savefig(os.path.join(DIR_GRAFICOS, 'temporal_todos_procedimentos_comparativo.png'), 
            dpi=600, bbox_inches='tight')
plt.show()

display(Markdown(f"**Gráfico comparativo salvo:** `temporal_todos_procedimentos_comparativo.png`"))

In [ ]:
#ANÁLISE ESPACIAL E DESIGUALDADES

In [ ]:
# Carregar dataset limpo
CAMINHO_DATASET_LIMPO = os.path.join(DIR_DADOS, "dataset_limpo.parquet")
df = pd.read_parquet(CAMINHO_DATASET_LIMPO)

logger.info(f"Dataset carregado para análise espacial: {len(df):,} registros")

POPULACAO_CENSO_2022 = {
    'RO': 1581016, 'AC': 830026, 'AM': 3941175, 'RR': 636303,
    'PA': 8116132, 'AP': 733508, 'TO': 1511459,
    'MA': 6775152, 'PI': 3269200, 'CE': 8791688,
    'RN': 3302406, 'PB': 3930885, 'PE': 9058155,
    'AL': 3127511, 'SE': 2209558, 'BA': 14136417,
    'MG': 20538718, 'ES': 3833486, 'RJ': 16054524,
    'SP': 44420459, 'PR': 11443208, 'SC': 7609601,
    'RS': 10880506, 'MS': 2756700, 'MT': 3658813,
    'GO': 7055228, 'DF': 2817068
}

# Validação: todas as 27 UFs devem estar presentes
assert len(POPULACAO_CENSO_2022) == 27

# Mapeamento de siglas
NOMES_UF = {
    'AC': 'Acre', 'AL': 'Alagoas', 'AP': 'Amapá', 'AM': 'Amazonas',
    'BA': 'Bahia', 'CE': 'Ceará', 'DF': 'Distrito Federal', 'ES': 'Espírito Santo',
    'GO': 'Goiás', 'MA': 'Maranhão', 'MT': 'Mato Grosso', 'MS': 'Mato Grosso do Sul',
    'MG': 'Minas Gerais', 'PA': 'Pará', 'PB': 'Paraíba', 'PR': 'Paraná',
    'PE': 'Pernambuco', 'PI': 'Piauí', 'RJ': 'Rio de Janeiro', 'RN': 'Rio Grande do Norte',
    'RS': 'Rio Grande do Sul', 'RO': 'Rondônia', 'RR': 'Roraima', 'SC': 'Santa Catarina',
    'SP': 'São Paulo', 'SE': 'Sergipe', 'TO': 'Tocantins'
}

ORDEM_REGIOES = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']

logger.info(f"Dados do Censo IBGE 2022 carregados: {sum(POPULACAO_CENSO_2022.values()):,} habitantes no Brasil")

In [ ]:
display(Markdown("Taxas de Procedimentos por 100.000 Habitantes"))
display(Markdown(f"**População de referência:** {sum(POPULACAO_CENSO_2022.values()):,} habitantes (Censo IBGE 2022)"))
display(Markdown(f"**Período dos procedimentos:** {df['ANO_CMPT'].min():.0f} - {df['ANO_CMPT'].max():.0f}"))

def calcular_taxas_por_100k(df, populacao_ref=POPULACAO_CENSO_2022, col_uf='UF_SIGLA', col_ano='ANO_CMPT'):
    # Agrupar por ano e UF
    contagem = df.groupby([col_ano, col_uf]).size().reset_index(name='N_PROCEDIMENTOS')
    
    # Adicionar população de referência (Censo 2022)
    contagem['POPULACAO_2022'] = contagem[col_uf].map(populacao_ref)
    
    # Calcular taxa por 100.000 habitantes
    contagem['TAXA_100K'] = (
        contagem['N_PROCEDIMENTOS'] / contagem['POPULACAO_2022'] * 100000
    ).round(2)
    
    # Adicionar nome do estado
    contagem['NOME_UF'] = contagem[col_uf].map(NOMES_UF)
    
    return contagem

# Calcular taxas para todo o período
df_taxas = calcular_taxas_por_100k(df)

# Taxas agregadas por UF (todo o período 2016-2025)
df_taxas_agregado = df_taxas.groupby('UF_SIGLA').agg({
    'NOME_UF': 'first',
    'N_PROCEDIMENTOS': 'sum',
    'POPULACAO_2022': 'first',
    'TAXA_100K': 'mean'
}).reset_index()

# Calcular taxa global por UF (soma de procedimentos / população 2022 * 100k)
df_taxas_agregado['TAXA_GLOBAL_100K'] = (
    df_taxas_agregado['N_PROCEDIMENTOS'] / df_taxas_agregado['POPULACAO_2022'] * 100000
).round(2)

# Ordenar por taxa global decrescente
df_taxas_agregado = df_taxas_agregado.sort_values('TAXA_GLOBAL_100K', ascending=False)

# Exibir top 15 UFs
display(Markdown("### Top 15 UFs por Taxa de Procedimentos (2016-2025)"))
display(df_taxas_agregado[['UF_SIGLA', 'NOME_UF', 'N_PROCEDIMENTOS', 'POPULACAO_2022', 'TAXA_GLOBAL_100K']].head(15).style.format({
    'N_PROCEDIMENTOS': '{:,.0f}',
    'POPULACAO_2022': '{:,.0f}',
    'TAXA_GLOBAL_100K': '{:.2f}'
}).background_gradient(subset=['TAXA_GLOBAL_100K'], cmap='YlOrRd'))

# Salvar tabela completa
df_taxas_agregado.to_csv(
    os.path.join(DIR_TABELAS, "taxas_procedimentos_censo2022.csv"),
    index=False,
    encoding='utf-8'
)
logger.info("Tabela de taxas salva")

In [ ]:
display(Markdown("Medida de Desigualdade na Distribuição"))

def calcular_gini(valores):
    """
    Calcula o Índice de Gini
    Fórmula: G = Σ(2i - n - 1) * x_i / (n * Σx_i)
    """
    valores = np.array(valores).flatten()
    valores = valores[~np.isnan(valores)]
    
    if len(valores) == 0 or np.sum(valores) == 0:
        return np.nan
    
    n = len(valores)
    valores_ordenados = np.sort(valores)
    indices = np.arange(1, n + 1)
    
    gini = np.sum((2 * indices - n - 1) * valores_ordenados) / (n * np.sum(valores_ordenados))
    return gini

def interpretar_gini(g):
    if pd.isna(g):
        return "Dados insuficientes"
    elif g < 0.2:
        return "Distribuição muito equitativa"
    elif g < 0.4:
        return "Desigualdade moderada"
    elif g < 0.6:
        return "Desigualdade alta"
    else:
        return "Desigualdade muito alta"

# Calcular Gini para distribuição absoluta de procedimentos
gini_absoluto = calcular_gini(df_taxas_agregado['N_PROCEDIMENTOS'])

# Calcular Gini para distribuição de taxas (ajustado por população)
gini_taxas = calcular_gini(df_taxas_agregado['TAXA_GLOBAL_100K'])

display(Markdown(f"""
| Métrica | Valor do Índice de Gini | Interpretação |
|---------|------------------------|---------------|
| **Distribuição Absoluta** (N de procedimentos) | {gini_absoluto:.3f} | {interpretar_gini(gini_absoluto)} |
| **Distribuição Ajustada** (Taxa por 100k hab) | {gini_taxas:.3f} | {interpretar_gini(gini_taxas)} |

**Nota:** Índice de Gini varia de 0 (equidade perfeita) a 1 (concentração máxima).
"""))

# Evolução temporal do Gini
gini_temporal = []
for ano in sorted(df['ANO_CMPT'].unique()):
    df_ano = df[df['ANO_CMPT'] == ano]
    contagem_ano = df_ano.groupby('UF_SIGLA').size()
    g_ano = calcular_gini(contagem_ano.values)
    gini_temporal.append({'ANO': ano, 'Gini': g_ano})

df_gini_temporal = pd.DataFrame(gini_temporal)

# Salvar resultados do Gini
resultados_gini = {
    'gini_absoluto': gini_absoluto,
    'gini_taxas': gini_taxas,
    'interpretacao_absoluto': interpretar_gini(gini_absoluto),
    'interpretacao_taxas': interpretar_gini(gini_taxas)
}
pd.DataFrame([resultados_gini]).to_csv(
    os.path.join(DIR_DADOS, "indice_gini_resultados.csv"),
    index=False,
    encoding='utf-8'
)

In [ ]:
display(Markdown("Mapa de Distribuição Geográfica das Taxas"))

# Caminho do shapefile do IBGE
SHAPEFILE_PATH = " "

if os.path.exists(SHAPEFILE_PATH):
    try:
        # Carregar shapefile do IBGE
        gdf_brasil = gpd.read_file(SHAPEFILE_PATH)
        
        # Garantir que a coluna de sigla está no formato correto
        if 'SIGLA_UF' in gdf_brasil.columns:
            gdf_brasil['UF'] = gdf_brasil['SIGLA_UF'].str.strip()
        elif 'UF' in gdf_brasil.columns:
            gdf_brasil['UF'] = gdf_brasil['UF'].str.strip()
        else:
            raise ValueError("Coluna de sigla da UF não encontrada no shapefile")
        
        # Merge com dados de taxas
        gdf_mapa = gdf_brasil.merge(
            df_taxas_agregado[['UF_SIGLA', 'NOME_UF', 'N_PROCEDIMENTOS', 'TAXA_GLOBAL_100K']],
            left_on='UF',
            right_on='UF_SIGLA',
            how='left'
        )
        
        # Preencher valores ausentes com 0 
        gdf_mapa['N_PROCEDIMENTOS'] = gdf_mapa['N_PROCEDIMENTOS'].fillna(0)
        gdf_mapa['TAXA_GLOBAL_100K'] = gdf_mapa['TAXA_GLOBAL_100K'].fillna(0)
        
        # Criar figura 
        fig, ax = plt.subplots(1, 1, figsize=(14, 12), dpi=600)
        
        # Plotar mapa coroplético
        gdf_mapa.plot(
            column='TAXA_GLOBAL_100K',
            cmap='viridis',
            linewidth=0.5,
            edgecolor='white',
            legend=True,
            ax=ax,
            legend_kwds={
                'label': 'Taxa por 100.000 habitantes\n(Censo IBGE 2022)',
                'orientation': 'vertical',
                'pad': 0.02
            }
        )
        
        # Adicionar rótulos das UFs com maiores taxas
        for idx, row in gdf_mapa.iterrows():
            if row['TAXA_GLOBAL_100K'] > df_taxas_agregado['TAXA_GLOBAL_100K'].quantile(0.75):
                ax.text(
                    row.geometry.centroid.x,
                    row.geometry.centroid.y,
                    row['UF'],
                    ha='center',
                    va='center',
                    fontsize=8,
                    fontweight='bold',
                    color='white',
                    bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.2')
                )
        
        ax.set_title(
            f'Taxa de Cirurgias Bariátricas por 100.000 Habitantes por UF, Brasil, 2016-2025 (n={len(df):,}; denom: Censo IBGE 2022)',
            fontsize=13,
            fontweight='bold',
            pad=20
        )
        
        ax.set_axis_off()
        plt.tight_layout()
        
        # Salvar em alta resolução
        caminho_mapa = os.path.join(DIR_MAPAS, 'mapa_taxas_censo2022_600dpi.png')
        plt.savefig(caminho_mapa, dpi=600, bbox_inches='tight', facecolor='white')
        plt.show()
        
        logger.info(f"Mapa coroplético salvo: {caminho_mapa}")
        
    except Exception as e:
        logger.error(f"Erro ao gerar mapa: {str(e)}")
        display(Markdown(f"**Erro ao carregar shapefile:** {str(e)}"))
        display(Markdown("Prosseguindo com visualização alternativa em gráfico de barras..."))
else:
    display(Markdown(f"Shapefile não encontrado em: {SHAPEFILE_PATH}"))
    display(Markdown("Utilizando visualização alternativa..."))

# Gráfico de barras horizontal por UF
plt.figure(figsize=(12, 14), dpi=600)

# Ordenar dados para plotagem
df_plot = df_taxas_agregado.sort_values('TAXA_GLOBAL_100K').tail(15)

# Criar barras horizontais
bars = plt.barh(df_plot['NOME_UF'], df_plot['TAXA_GLOBAL_100K'], 
                color=plt.cm.viridis(np.linspace(0.2, 0.9, len(df_plot))),
                edgecolor='black', alpha=0.85)

# Adicionar valores no final das barras
for bar, taxa in zip(bars, df_plot['TAXA_GLOBAL_100K']):
    plt.text(taxa + 0.5, bar.get_y() + bar.get_height()/2, 
             f'{taxa:.1f}', va='center', fontsize=9, fontweight='bold')

# Configurar gráfico
plt.xlabel('Taxa por 100.000 Habitantes', fontsize=12, fontweight='bold')
plt.ylabel('Unidade Federativa', fontsize=12, fontweight='bold')
plt.title(f'Top 15 UFs por Taxa de Cirurgias Bariátricas, Brasil, 2016-2025 (n={len(df):,})',
          fontsize=14, fontweight='bold', pad=15)
plt.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()

caminho_grafico = os.path.join(DIR_GRAFICOS, 'barras_taxas_uf_top15.png')
plt.savefig(caminho_grafico, dpi=600, bbox_inches='tight')
plt.show()

logger.info(f"Gráfico de barras salvo: {caminho_grafico}")

In [ ]:
display(Markdown("Desigualdade Regional - Agregação por Macrorregião"))

# Adicionar coluna de região ao dataset de taxas
def obter_regiao(uf):
    regioes = {
        'Norte': ['RO', 'AC', 'AM', 'RR', 'PA', 'AP', 'TO'],
        'Nordeste': ['MA', 'PI', 'CE', 'RN', 'PB', 'PE', 'AL', 'SE', 'BA'],
        'Centro-Oeste': ['MS', 'MT', 'GO', 'DF'],
        'Sudeste': ['SP', 'RJ', 'MG', 'ES'],
        'Sul': ['PR', 'SC', 'RS']
    }
    for regiao, ufs in regioes.items():
        if uf in ufs:
            return regiao
    return 'Desconhecida'

df_taxas_agregado['REGIAO'] = df_taxas_agregado['UF_SIGLA'].apply(obter_regiao)

# Agregar por região
df_regiao = df_taxas_agregado.groupby('REGIAO').agg({
    'N_PROCEDIMENTOS': 'sum',
    'POPULACAO_2022': 'sum',
    'UF_SIGLA': 'count'
}).reset_index()

# Calcular taxa regional e percentual do total nacional
df_regiao['TAXA_REGIONAL_100K'] = (
    df_regiao['N_PROCEDIMENTOS'] / df_regiao['POPULACAO_2022'] * 100000
).round(2)
df_regiao['PERCENTUAL_NACIONAL'] = (
    df_regiao['N_PROCEDIMENTOS'] / df_regiao['N_PROCEDIMENTOS'].sum() * 100
).round(1)

# Ordenar por região geográfica
df_regiao['ORDEM'] = df_regiao['REGIAO'].apply(
    lambda x: ORDEM_REGIOES.index(x) if x in ORDEM_REGIOES else 99
)
df_regiao = df_regiao.sort_values('ORDEM').drop('ORDEM', axis=1)

display(Markdown("### Distribuição Regional de Procedimentos Bariátricos"))
display(df_regiao.style.format({
    'N_PROCEDIMENTOS': '{:,.0f}',
    'POPULACAO_2022': '{:,.0f}',
    'TAXA_REGIONAL_100K': '{:.2f}',
    'PERCENTUAL_NACIONAL': '{:.1f}%',
    'UF_SIGLA': '{:.0f}'
}).background_gradient(subset=['TAXA_REGIONAL_100K'], cmap='YlOrRd'))

# Salvar tabela regional
df_regiao.to_csv(
    os.path.join(DIR_TABELAS, "distribuicao_regional.csv"),
    index=False,
    encoding='utf-8'
)

# Visualização: Gráfico de barras por região
plt.figure(figsize=(10, 6), dpi=600)

# Cores por região (paleta acessível)
cores_regiao = {
    'Norte': '#1f77b4',
    'Nordeste': '#ff7f0e',
    'Centro-Oeste': '#2ca02c',
    'Sudeste': '#d62728',
    'Sul': '#9467bd'
}

# Plotar barras
for idx, row in df_regiao.iterrows():
    plt.bar(
        row['REGIAO'],
        row['PERCENTUAL_NACIONAL'],
        color=cores_regiao.get(row['REGIAO'], '#7f7f7f'),
        edgecolor='black',
        alpha=0.85,
        label=f"{row['REGIAO']}: {row['PERCENTUAL_NACIONAL']:.1f}% ({row['N_PROCEDIMENTOS']:,.0f} proc.)"
    )

# Configurar gráfico
plt.ylabel('Percentual do Total Nacional (%)', fontsize=12, fontweight='bold')
plt.xlabel('Região', fontsize=12, fontweight='bold')
plt.title(f'Distribuição Regional de Cirurgias Bariátricas, Brasil, 2016-2025 (n={len(df):,})',
          fontsize=14, fontweight='bold', pad=15)
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.legend(loc='upper right', fontsize=9)
plt.tight_layout()

caminho_regiao = os.path.join(DIR_GRAFICOS, 'distribuicao_regional_barras.png')
plt.savefig(caminho_regiao, dpi=600, bbox_inches='tight')
plt.show()

logger.info(f"Gráfico regional salvo: {caminho_regiao}")

In [ ]:
display(Markdown("Validação e Resumo Resultados Espaciais"))

# Resumo dos indicadores calculados
resumo_espacial = {
    'Total de procedimentos analisados': len(df),
    'UFs com dados': df_taxas_agregado['UF_SIGLA'].nunique(),
    'Período': f"{df['ANO_CMPT'].min():.0f} - {df['ANO_CMPT'].max():.0f}",
    'Gini (distribuição absoluta)': f"{gini_absoluto:.3f} - {interpretar_gini(gini_absoluto)}",
    'Gini (taxas ajustadas)': f"{gini_taxas:.3f} - {interpretar_gini(gini_taxas)}",
    'Região com maior taxa': df_regiao.loc[df_regiao['TAXA_REGIONAL_100K'].idxmax(), 'REGIAO'],
    'Região com menor taxa': df_regiao.loc[df_regiao['TAXA_REGIONAL_100K'].idxmin(), 'REGIAO'],
    'UF com maior taxa': df_taxas_agregado.iloc[0]['NOME_UF'],
    'UF com menor taxa': df_taxas_agregado.iloc[-1]['NOME_UF']
}

display(pd.DataFrame(list(resumo_espacial.items()), columns=['Indicador', 'Valor']).style.set_properties(**{'text-align': 'left'}))

# Exportar todos os resultados espaciais
resultados_espaciais = {
    'taxas_por_uf': df_taxas_agregado,
    'distribuicao_regional': df_regiao,
    'indice_gini': resultados_gini
}

# Salvar em arquivo consolidado
with pd.ExcelWriter(os.path.join(DIR_TABELAS, "resultados_espaciais_consolidados.xlsx"), 
                    engine='openpyxl') as writer:
    for sheet_name, df_result in resultados_espaciais.items():
        if df_result is not None and isinstance(df_result, pd.DataFrame):
            sheet_name_clean = sheet_name[:31].replace('/', '_')
            df_result.to_excel(writer, sheet_name=sheet_name_clean, index=False)

logger.info("Resultados espaciais exportados com sucesso")

print(f"\nAnálise Espacial e Desigualdades - CONCLUÍDO")
print(f"Mapas e gráficos salvos em: {os.path.abspath(DIR_MAPAS)}")
print(f"Tabelas salvas em: {os.path.abspath(DIR_TABELAS)}")

In [ ]:
#Shapefile e dados base
SHAPEFILE_PATH = "" #caminho shapefile

POPULACAO_CENSO_2022 = {
    'RO': 1581016, 'AC': 830026,  'AM': 3941175, 'RR': 636303,
    'PA': 8116132, 'AP': 733508,  'TO': 1511459,
    'MA': 6775152, 'PI': 3269200, 'CE': 8791688,
    'RN': 3302406, 'PB': 3930885, 'PE': 9058155,
    'AL': 3127511, 'SE': 2209558, 'BA': 14136417,
    'MG': 20538718,'ES': 3833486, 'RJ': 16054524,
    'SP': 44420459,'PR': 11443208,'SC': 7609601,
    'RS': 10880506,'MS': 2756700, 'MT': 3658813,
    'GO': 7055228, 'DF': 2817068,
}

gdf_brasil = gpd.read_file(SHAPEFILE_PATH, engine="pyogrio")

dados_uf = df.groupby('UF_SIGLA').size().reset_index(name='N_Total')
dados_uf['POPULACAO'] = dados_uf['UF_SIGLA'].map(POPULACAO_CENSO_2022)
dados_uf['TAXA_100K'] = (dados_uf['N_Total'] / dados_uf['POPULACAO']) * 100_000

display(dados_uf.sort_values('N_Total', ascending=False))

In [ ]:
#Mapa 1: Contagem total de procedimentos por UF
gdf1 = gdf_brasil.merge(dados_uf, left_on='SIGLA_UF', right_on='UF_SIGLA', how='left')
gdf1['N_Total'] = gdf1['N_Total'].fillna(0)
vmin1, vmax1 = 0, gdf1['N_Total'].max()

fig, ax = plt.subplots(figsize=(14, 12), dpi=200)
fig.patch.set_facecolor('#F7F7F5')
ax.set_facecolor('#F7F7F5')

gdf1.plot(
    column='N_Total', cmap='YlOrRd',
    linewidth=0.6, edgecolor='#FFFFFF',
    vmin=vmin1, vmax=vmax1,
    legend=True, ax=ax,
    legend_kwds=dict(label='Número de Procedimentos',
                 orientation='vertical', pad=0.01, shrink=0.7,
                 format='{x:,.0f}')
)

norm1 = Normalize(vmin=vmin1, vmax=vmax1)
cmap1 = cm.get_cmap('YlOrRd')

for _, row in gdf1.iterrows():
    if row.geometry is None or np.isnan(row['N_Total']) or row['N_Total'] == 0:
        continue
    cx = row.geometry.centroid.x
    cy = row.geometry.centroid.y
    r, g, b, _ = cmap1(norm1(row['N_Total']))
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    tc = 'white' if lum < 0.55 else 'black'
    ax.text(cx, cy + 0.15, row['SIGLA_UF'],
            ha='center', va='center',
            fontsize=7.5, fontweight='bold', color=tc, zorder=5)
    ax.text(cx, cy - 0.35, f"{int(row['N_Total']):,}".replace(',', '.'),
            ha='center', va='center',
            fontsize=6.2, color=tc, alpha=0.92, zorder=5)

ax.set_axis_off()
ax.set_title(f'Distribuição de Cirurgias Bariátricas por UF\nBrasil, 2016–2025 (n={len(df):,})',
             fontsize=13, fontweight='bold', pad=16)
plt.tight_layout()
plt.savefig(os.path.join(DIR_MAPAS, 'mapa_brasil_contagem_total.png'),
            dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

In [ ]:
#Mapa 2: Taxa por 100.000 habitantes
gdf2 = gdf_brasil.merge(dados_uf[['UF_SIGLA', 'TAXA_100K']],
                        left_on='SIGLA_UF', right_on='UF_SIGLA', how='left')
gdf2['TAXA_100K'] = gdf2['TAXA_100K'].fillna(0)
vmin2, vmax2 = 0, gdf2['TAXA_100K'].max()

fig, ax = plt.subplots(figsize=(14, 12), dpi=200)
fig.patch.set_facecolor('#F7F7F5')
ax.set_facecolor('#F7F7F5')

gdf2.plot(
    column='TAXA_100K', cmap='RdYlBu_r',
    linewidth=0.6, edgecolor='#FFFFFF',
    vmin=vmin2, vmax=vmax2,
    legend=True, ax=ax,
    legend_kwds=dict(label='Taxa por 100.000 hab.',
                 orientation='vertical', pad=0.01, shrink=0.7,
                 format='{x:.1f}')
)

norm2 = Normalize(vmin=vmin2, vmax=vmax2)
cmap2 = cm.get_cmap('RdYlBu_r')

for _, row in gdf2.iterrows():
    if row.geometry is None or np.isnan(row['TAXA_100K']) or row['TAXA_100K'] == 0:
        continue
    cx = row.geometry.centroid.x
    cy = row.geometry.centroid.y
    r, g, b, _ = cmap2(norm2(row['TAXA_100K']))
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    tc = 'white' if lum < 0.55 else 'black'
    ax.text(cx, cy + 0.15, row['SIGLA_UF'],
            ha='center', va='center',
            fontsize=7.5, fontweight='bold', color=tc, zorder=5)
    ax.text(cx, cy - 0.35, f"{row['TAXA_100K']:.1f}",
            ha='center', va='center',
            fontsize=6.2, color=tc, alpha=0.92, zorder=5)

ax.set_axis_off()
ax.set_title('Taxa de Cirurgias Bariátricas por 100.000 Habitantes por UF\nBrasil, 2016–2025',
             fontsize=13, fontweight='bold', pad=16)
plt.tight_layout()
plt.savefig(os.path.join(DIR_MAPAS, 'mapa_brasil_taxa_100k.png'),
            dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

In [ ]:
# Mapa 3: Custo médio por UF
if 'VAL_TOT' in df.columns:
    custos_uf = df.groupby('UF_SIGLA')['VAL_TOT'].mean().reset_index(name='CUSTO_MEDIO')
    gdf3 = gdf_brasil.merge(custos_uf, left_on='SIGLA_UF', right_on='UF_SIGLA', how='left')
    gdf3['CUSTO_MEDIO'] = gdf3['CUSTO_MEDIO'].fillna(0)
    vmin3, vmax3 = 0, gdf3['CUSTO_MEDIO'].max()

    fig, ax = plt.subplots(figsize=(14, 12), dpi=200)
    fig.patch.set_facecolor('#F7F7F5')
    ax.set_facecolor('#F7F7F5')

    gdf3.plot(
        column='CUSTO_MEDIO', cmap='OrRd',
        linewidth=0.6, edgecolor='#FFFFFF',
        vmin=vmin3, vmax=vmax3,
        legend=True, ax=ax,
        legend_kwds=dict(label='Custo Médio (R$)',
                 orientation='vertical', pad=0.01, shrink=0.7,
                 format='R$ {x:,.0f}')
    )

    norm3 = Normalize(vmin=vmin3, vmax=vmax3)
    cmap3 = cm.get_cmap('OrRd')

    for _, row in gdf3.iterrows():
        if row.geometry is None or np.isnan(row['CUSTO_MEDIO']) or row['CUSTO_MEDIO'] == 0:
            continue
        cx = row.geometry.centroid.x
        cy = row.geometry.centroid.y
        r, g, b, _ = cmap3(norm3(row['CUSTO_MEDIO']))
        lum = 0.299 * r + 0.587 * g + 0.114 * b
        tc = 'white' if lum < 0.55 else 'black'
        ax.text(cx, cy + 0.15, row['SIGLA_UF'],
                ha='center', va='center',
                fontsize=7.5, fontweight='bold', color=tc, zorder=5)
        ax.text(cx, cy - 0.35, f"R${row['CUSTO_MEDIO']:,.0f}".replace(',', '.'),
                ha='center', va='center',
                fontsize=6.0, color=tc, alpha=0.92, zorder=5)

    ax.set_axis_off()
    ax.set_title('Custo Médio por Procedimento por UF\nBrasil, 2016–2025',
                 fontsize=13, fontweight='bold', pad=16)
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_MAPAS, 'mapa_brasil_custo_medio.png'),
                dpi=200, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

In [ ]:
#ANÁLISE INFERENCIAL E COMPARATIVA

In [ ]:
CAMINHO_DATASET_LIMPO = os.path.join(DIR_DADOS, "dataset_limpo.parquet")
df = pd.read_parquet(CAMINHO_DATASET_LIMPO)

logger.info(f"Dataset carregado para análise inferencial: {len(df):,} registros")


In [ ]:
def formatar_pvalor(p):
    """
    Formato: p-valor<0,001
    """
    if pd.isna(p):
        return 'p-valor=NA'
    elif p < 0.001:
        return 'p-valor<0,001'
    else:
        return f'p-valor={p:.3f}'.replace('.', ',')


In [ ]:
def calcular_cramers_v(chi2, n, r, c):
    """Calcula V de Cramer para teste Qui-Quadrado"""
    return np.sqrt(chi2 / (n * (min(r, c) - 1)))

def calcular_eta_squared(ss_between, ss_total):
    """Calcula Eta-squared para ANOVA"""
    return ss_between / ss_total

def calcular_epsilon_squared(h, n, k):
    """Calcula Epsilon-squared para Kruskal-Wallis"""
    return (h - k + 1) / (n - k)

logger.info("Funções de formatação e tamanho de efeito carregadas")

In [ ]:
display(Markdown("Validação para Variáveis Contínuas"))

# Variáveis contínuas
VAR_CONTINUAS = {
    'IDADE': 'Idade (anos)',
    'DIAS_PERM': 'Dias de Permanência',
    'VAL_TOT': 'Custo Total (R$)'
}

# Grupos para comparação
GRUPOS_PROC = df['NOME_PROCEDIMENTO'].unique()

display(Markdown(f"**Variáveis analisadas:** {len(VAR_CONTINUAS)}"))
display(Markdown(f"**Grupos para comparação:** {len(GRUPOS_PROC)} procedimentos"))


In [ ]:
# TESTE DE NORMALIDADE (Shapiro-Wilk + Kolmogorov-Smirnov) + VISUALIZAÇÃO

display(Markdown("### Teste de Normalidade por Variável e Procedimento"))

resultados_normalidade = []

for var, nome_var in VAR_CONTINUAS.items():
    if var not in df.columns:
        continue
    
    for proc in GRUPOS_PROC:
        dados = df[df['NOME_PROCEDIMENTO'] == proc][var].dropna()
        n_total = len(dados)
        
        if n_total < 10:
            continue
        
        # Escolher teste conforme tamanho da amostra
        if n_total <= 5000:
            estatistica, p_valor = shapiro(dados)
            teste_usado = 'Shapiro-Wilk'
        else:
            # KS test com parâmetros estimados dos dados
            estatistica, p_valor = kstest(dados, 'norm', args=(dados.mean(), dados.std()))
            teste_usado = 'Kolmogorov-Smirnov'
        
        normal = p_valor > 0.05
        
        resultados_normalidade.append({
            'Variavel': nome_var,
            'Procedimento': proc,
            'N_Total': n_total,
            'Teste': teste_usado,
            'Estatistica': round(estatistica, 4),
            'P_valor': round(p_valor, 4),
            'Normal': normal
        })
        
        # GRÁFICOS: Histograma + Q-Q Plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=600)
        
        # Histograma com curva normal teórica
        ax1.hist(dados, bins=40, density=True, alpha=0.7, 
                 color=CORES_PROCEDIMENTOS.get([k for k,v in CODIGOS_BARIATRICOS.items() if v==proc][0] if proc in CODIGOS_BARIATRICOS.values() else list(CORES_PROCEDIMENTOS.values())[0], '#3498db'),
                 edgecolor='black', linewidth=0.5)
        
        mu, sigma = dados.mean(), dados.std()
        x = np.linspace(dados.min(), dados.max(), 100)
        ax1.plot(x, norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal Teórica')
        ax1.set_xlabel(nome_var, fontweight='bold')
        ax1.set_ylabel('Densidade', fontweight='bold')
        ax1.set_title(f'Histograma\nn={n_total:,}', fontweight='bold', fontsize=10)
        ax1.legend(fontsize=9)
        ax1.grid(alpha=0.3, linestyle='--')
        
        # Q-Q Plot
        probplot(dados, dist="norm", plot=ax2)
        ax2.set_title('Q-Q Plot', fontweight='bold', fontsize=10)
        ax2.grid(alpha=0.3, linestyle='--')
        
        plt.suptitle(f'{nome_var} - {proc} | {teste_usado}: estat={estatistica:.4f}, p={p_valor:.4f}', 
                     fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        
        # Salvar gráfico
        nome_proc = proc.replace('/', '_').replace(' ', '_')
        nome_var_clean = nome_var.replace('/', '_').replace(' ', '_').replace('(', '').replace(')', '')
        plt.savefig(os.path.join(DIR_GRAFICOS, f'normalidade_{nome_var_clean}_{nome_proc}.png'), 
                    dpi=600, bbox_inches='tight')
        plt.show()


In [ ]:
# TABELA DE RESULTADOS

df_normalidade = pd.DataFrame(resultados_normalidade)

display(Markdown("**Resultados do Teste de Normalidade**"))

df_exibir = df_normalidade.copy()
df_exibir['P_valor'] = df_exibir['P_valor'].apply(lambda p: f'{p:.4f}')
df_exibir['Estatistica'] = df_exibir['Estatistica'].apply(lambda s: f'{s:.4f}')
df_exibir['Normal'] = df_exibir['Normal'].apply(lambda x: ' Sim' if x else ' Não')

display(df_exibir[['Procedimento', 'Variavel', 'N_Total', 'Teste', 'Estatistica', 'P_valor', 'Normal']]
        .style.set_properties(**{'text-align': 'center'})
        .applymap(lambda x: 'background-color: #d4edda' if x == ' Sim' 
                  else 'background-color: #f8d7da' if x == ' Não' 
                  else '', subset=['Normal']))

# Salvar resultados
df_normalidade.to_csv(
    os.path.join(DIR_DADOS, "teste_normalidade.csv"),
    index=False, encoding='utf-8'
)
logger.info("Teste de normalidade concluído")

In [ ]:
# TESTE DE HOMOCEDASTICIDADE (Levene)
display(Markdown("Teste de Homogeneidade de Variâncias (Levene)"))

resultados_levene = []

for var, nome_var in VAR_CONTINUAS.items():
    if var not in df.columns:
        continue
    
    # Preparar grupos para Levene
    grupos = [df[df['NOME_PROCEDIMENTO'] == proc][var].dropna().values 
              for proc in GRUPOS_PROC 
              if len(df[df['NOME_PROCEDIMENTO'] == proc][var].dropna()) >= 2]
    
    if len(grupos) >= 2:
        stat, p_valor = levene(*grupos)
        homogenea = p_valor > 0.05
        
        resultados_levene.append({
            'Variavel': nome_var,
            'Estatistica_F': stat,
            'P_valor': p_valor,
            'Homogenea': homogenea
        })

df_levene = pd.DataFrame(resultados_levene)

display(Markdown("**Resultados do Teste de Levene por Variável**"))
display(df_levene.style.format({
    'Estatistica_F': '{:.4f}',
    'P_valor': '{:.4f}'
}).applymap(lambda x: 'background-color: #d4edda' if x == True 
            else 'background-color: #f8d7da' if x == False 
            else '', subset=['Homogenea']))

# Salvar resultados
df_levene.to_csv(
    os.path.join(DIR_DADOS, "teste_homocedasticidade_levene.csv"),
    index=False,
    encoding='utf-8'
)


In [ ]:
#TESTE PARAMÉTRICO OU NÃO-PARAMÉTRICO
display(Markdown("Decisão Analítica por Variável"))

decisao_testes = []

for var, nome_var in VAR_CONTINUAS.items():
    # Verificar normalidade
    normalidade_var = df_normalidade[df_normalidade['Variavel'] == nome_var]['Normal'].all() if len(df_normalidade[df_normalidade['Variavel'] == nome_var]) > 0 else False
    
    # Verificar homocedasticidade
    homogenea_var = df_levene[df_levene['Variavel'] == nome_var]['Homogenea'].values[0] if len(df_levene[df_levene['Variavel'] == nome_var]) > 0 else False
    
    # Decisão
    if normalidade_var and homogenea_var:
        teste_recomendado = 'ANOVA One-Way + Tukey HSD'
        tipo = 'Paramétrico'
    elif normalidade_var and not homogenea_var:
        teste_recomendado = 'Welch ANOVA + Games-Howell'
        tipo = 'Paramétrico Robusto'
    else:
        teste_recomendado = 'Kruskal-Wallis + Dunn-Bonferroni'
        tipo = 'Não-paramétrico'
    
    decisao_testes.append({
        'Variavel': nome_var,
        'Normalidade': normalidade_var,
        'Homocedasticidade': homogenea_var,
        'Teste_Recomendado': teste_recomendado,
        'Tipo': tipo
    })

df_decisao = pd.DataFrame(decisao_testes)

display(df_decisao.style.applymap(
    lambda x: 'background-color: #2ecc71; color: white' if 'Paramétrico' in str(x)
    else 'background-color: #e74c3c; color: white' if 'Não-paramétrico' in str(x)
    else '', subset=['Tipo']
))

logger.info("Validação de pressupostos concluída")

In [ ]:
display(Markdown("Comparação de Variáveis Contínuas entre Procedimentos"))

resultados_inferenciais = []
tabelas_posthoc = {}

for var, nome_var in VAR_CONTINUAS.items():
    if var not in df.columns:
        continue
    
    display(Markdown(f"### 2.{list(VAR_CONTINUAS.keys()).index(var) + 1} {nome_var}"))
    
    # Preparar grupos
    grupos = {}
    for proc in GRUPOS_PROC:
        dados = df[df['NOME_PROCEDIMENTO'] == proc][var].dropna()
        if len(dados) >= 5:
            grupos[proc] = dados
    
    if len(grupos) < 2:
        display(Markdown("*Dados insuficientes para comparação estatística*"))
        continue
    # TESTE PRINCIPAL (ANOVA ou Kruskal-Wallis)
    decisao_var = df_decisao[df_decisao['Variavel'] == nome_var].iloc[0]
    
    if 'ANOVA' in decisao_var['Teste_Recomendado']:
        # ANOVA One-Way
        display(Markdown("**Teste Aplicado:** ANOVA One-Way"))
        
        valores_grupos = list(grupos.values())
        stat, p_valor = f_oneway(*valores_grupos)
        
        # Calcular Eta-squared (tamanho de efeito)
        modelo_anova = ols(f'{var} ~ C(NOME_PROCEDIMENTO)', data=df).fit()
        tabela_anova = anova_lm(modelo_anova)
        ss_between = tabela_anova['sum_sq']['C(NOME_PROCEDIMENTO)']
        ss_total = tabela_anova['sum_sq'].sum()
        eta_sq = calcular_eta_squared(ss_between, ss_total)
        
        # Interpretar tamanho de efeito
        if eta_sq < 0.01:
            interpretacao_eta = 'Desprezível'
        elif eta_sq < 0.06:
            interpretacao_eta = 'Pequeno'
        elif eta_sq < 0.14:
            interpretacao_eta = 'Médio'
        else:
            interpretacao_eta = 'Grande'
        
        display(Markdown(f"""
| Estatística | Valor |
|-------------|-------|
| **F** | {stat:.4f} |
| **Graus de liberdade** | ({len(grupos)-1}, {sum(len(g) for g in valores_grupos)-len(grupos)}) |
| {formatar_pvalor(p_valor)} | {p_valor:.4f} |
| **η² (Eta-squared)** | {eta_sq:.4f} ({interpretacao_eta}) |
"""))
        
        # Post-hoc: Tukey HSD
        if p_valor < 0.05:
            display(Markdown("**Teste Post-hoc:** Tukey HSD"))
            
            dados_tukey = []
            grupos_tukey = []
            for proc, dados in grupos.items():
                dados_tukey.extend(dados.values)
                grupos_tukey.extend([proc] * len(dados))
            
            mc = MultiComparison(dados_tukey, grupos_tukey)
            resultado_tukey = mc.tukeyhsd()
            
            # Converter para DataFrame
            df_tukey = pd.DataFrame(data=resultado_tukey.summary().data[1:],
                                   columns=resultado_tukey.summary().data[0])
            
            display(df_tukey.style.format({
                'meandiff': '{:.4f}',
                'p-adj': '{:.4f}',
                'lower': '{:.4f}',
                'upper': '{:.4f}'
            }).applymap(lambda x: 'background-color: #d4edda' if x == True 
                        else 'background-color: #f8d7da' if x == False 
                        else '', subset=['reject']))
            
            tabelas_posthoc[f'Tukey_{var}'] = df_tukey
        
        resultados_inferenciais.append({
            'Variavel': nome_var,
            'Teste': 'ANOVA One-Way',
            'Estatistica': stat,
            'Gl': f"({len(grupos)-1}, {sum(len(g) for g in valores_grupos)-len(grupos)})",
            'P_valor': p_valor,
            'P_valor_formatado': formatar_pvalor(p_valor),
            'Tamanho_Efeito': eta_sq,
            'Medida_Efeito': 'η²',
            'Significativo': p_valor < 0.05
        })
        
    else:
        # Kruskal-Wallis
        display(Markdown("**Teste Aplicado:** Kruskal-Wallis"))
        
        valores_grupos = list(grupos.values())
        stat, p_valor = kruskal(*valores_grupos)
        
        # Calcular Epsilon-squared (tamanho de efeito)
        h = stat
        n = sum(len(g) for g in valores_grupos)
        k = len(grupos)
        epsilon_sq = calcular_epsilon_squared(h, n, k)
        
        # Interpretar tamanho de efeito
        if epsilon_sq < 0.01:
            interpretacao_eps = 'Desprezível'
        elif epsilon_sq < 0.08:
            interpretacao_eps = 'Pequeno'
        elif epsilon_sq < 0.26:
            interpretacao_eps = 'Médio'
        else:
            interpretacao_eps = 'Grande'
        
        display(Markdown(f"""
| Estatística | Valor |
|-------------|-------|
| **H** | {stat:.4f} |
| **Graus de liberdade** | {len(grupos)-1} |
| {formatar_pvalor(p_valor)} | {p_valor:.4f} |
| **ε² (Epsilon-squared)** | {epsilon_sq:.4f} ({interpretacao_eps}) |
"""))
        
        # Post-hoc: Dunn-Bonferroni (Mann-Whitney pareado com correção)
        if p_valor < 0.05:
            display(Markdown("**Teste Post-hoc:** Mann-Whitney U com Correção de Bonferroni"))
            
            n_comparacoes = len(grupos) * (len(grupos) - 1) / 2
            alpha_corrigido = 0.05 / n_comparacoes
            
            comparacoes_mw = []
            procs = list(grupos.keys())
            
            for i in range(len(procs)):
                for j in range(i + 1, len(procs)):
                    stat_mw, p_mw = mannwhitneyu(grupos[procs[i]], grupos[procs[j]], 
                                                 alternative='two-sided')
                    p_ajustado = min(p_mw * n_comparacoes, 1.0)
                    significativo = p_ajustado < 0.05
                    
                    comparacoes_mw.append({
                        'Grupo_1': procs[i],
                        'Grupo_2': procs[j],
                        'Estatistica_U': stat_mw,
                        'P_valor_bruto': p_mw,
                        'P_valor_ajustado': p_ajustado,
                        'P_valor_formatado': formatar_pvalor(p_ajustado),
                        'Significativo': significativo
                    })
            
            df_mw = pd.DataFrame(comparacoes_mw)
            
            display(df_mw.style.format({
                'Estatistica_U': '{:.2f}',
                'P_valor_bruto': '{:.4f}',
                'P_valor_ajustado': '{:.4f}'
            }).applymap(lambda x: 'background-color: #d4edda' if x == True 
                        else 'background-color: #f8d7da' if x == False 
                        else '', subset=['Significativo']))
            
            tabelas_posthoc[f'Dunn_{var}'] = df_mw
        
        resultados_inferenciais.append({
            'Variavel': nome_var,
            'Teste': 'Kruskal-Wallis',
            'Estatistica': stat,
            'Gl': len(grupos)-1,
            'P_valor': p_valor,
            'P_valor_formatado': formatar_pvalor(p_valor),
            'Tamanho_Efeito': epsilon_sq,
            'Medida_Efeito': 'ε²',
            'Significativo': p_valor < 0.05
        })

# Salvar resultados inferenciais
df_resultados_inferenciais = pd.DataFrame(resultados_inferenciais)
df_resultados_inferenciais.to_csv(
    os.path.join(DIR_TABELAS, "resultados_inferenciais_continuas.csv"),
    index=False,
    encoding='utf-8',
    sep=';'
)

logger.info(f"Análises inferenciais concluídas para {len(resultados_inferenciais)} variáveis")

In [ ]:
display(Markdown("Comparação de Variáveis Categóricas entre Procedimentos"))

# Variáveis categóricas para análise
VAR_CATEGORICAS = {
    'SEXO_DESC': 'Sexo',
    'REGIAO': 'Região de Residência',
    'UF_SIGLA': 'UF de Residência'
}

resultados_categoricos = []

for var, nome_var in VAR_CATEGORICAS.items():
    if var not in df.columns:
        continue
    
    display(Markdown(f"### 3.{list(VAR_CATEGORICAS.keys()).index(var) + 1} {nome_var}"))
    
    # Criar tabela de contingência
    tabela_contingencia = pd.crosstab(df['NOME_PROCEDIMENTO'], df[var])
    
    # Verificar se há células com esperados < 5
    chi2, p_valor, dof, esperados = chi2_contingency(tabela_contingencia)
    
    celulas_esperados_baixos = (esperados < 5).sum()
    percentual_baixas = celulas_esperados_baixos / esperados.size * 100
    
    # Decisão: Qui-Quadrado ou Fisher
    if percentual_baixas > 20:
        display(Markdown(f"**Atenção:** {percentual_baixas:.1f}% das células têm esperados < 5"))
        display(Markdown("**Teste Aplicado:** Qui-Quadrado (com ressalva)"))
    else:
        display(Markdown("**Teste Aplicado:** Qui-Quadrado de Pearson"))
    
    # Calcular V de Cramer (tamanho de efeito)
    n = tabela_contingencia.sum().sum()
    r, c = tabela_contingencia.shape
    cramers_v = calcular_cramers_v(chi2, n, r, c)
    
    # Interpretar tamanho de efeito
    min_dim = min(r-1, c-1)
    if min_dim == 1:
        if cramers_v < 0.1:
            interpretacao_v = 'Desprezível'
        elif cramers_v < 0.3:
            interpretacao_v = 'Pequeno'
        elif cramers_v < 0.5:
            interpretacao_v = 'Médio'
        else:
            interpretacao_v = 'Grande'
    else:
        if cramers_v < 0.08:
            interpretacao_v = 'Desprezível'
        elif cramers_v < 0.22:
            interpretacao_v = 'Pequeno'
        elif cramers_v < 0.36:
            interpretacao_v = 'Médio'
        else:
            interpretacao_v = 'Grande'
    
    # Exibir tabela de contingência
    display(Markdown("**Tabela de Contingência**"))
    display(tabela_contingencia.style.format('{:,.0f}').background_gradient(
        cmap='YlGnBu', axis=None))
    
    # Exibir resultados do teste
    display(Markdown(f"""
| Estatística | Valor |
|-------------|-------|
| **χ² (Qui-Quadrado)** | {chi2:.4f} |
| **Graus de liberdade** | {dof} |
| {formatar_pvalor(p_valor)} | {p_valor:.4f} |
| **V de Cramer** | {cramers_v:.4f} ({interpretacao_v}) |
| **Células com esperados < 5** | {celulas_esperados_baixos} ({percentual_baixas:.1f}%) |
"""))
    
    resultados_categoricos.append({
        'Variavel': nome_var,
        'Teste': 'Qui-Quadrado',
        'Estatistica': chi2,
        'Gl': dof,
        'P_valor': p_valor,
        'P_valor_formatado': formatar_pvalor(p_valor),
        'Tamanho_Efeito': cramers_v,
        'Medida_Efeito': "V de Cramer",
        'Significativo': p_valor < 0.05,
        'Células_Esperados_Baixas': celulas_esperados_baixos
    })

# Salvar resultados categóricos
df_resultados_categoricos = pd.DataFrame(resultados_categoricos)
df_resultados_categoricos.to_csv(
    os.path.join(DIR_TABELAS, "resultados_inferenciais_categoricas.csv"),
    index=False,
    encoding='utf-8',
    sep=';'
)

logger.info(f"Análises inferenciais concluídas para {len(resultados_categoricos)} variáveis categóricas")

In [ ]:
display(Markdown("Visualização Integrada dos Tamanhos de Efeito"))

# Combinar todos os resultados
todos_efeitos = []

for _, row in df_resultados_inferenciais.iterrows():
    todos_efeitos.append({
        'Variavel': row['Variavel'],
        'Tipo': 'Contínua',
        'Medida': row['Medida_Efeito'],
        'Valor': row['Tamanho_Efeito'],
        'Significativo': row['Significativo'],
        'Teste': row['Teste']
    })

for _, row in df_resultados_categoricos.iterrows():
    todos_efeitos.append({
        'Variavel': row['Variavel'],
        'Tipo': 'Categórica',
        'Medida': row['Medida_Efeito'],
        'Valor': row['Tamanho_Efeito'],
        'Significativo': row['Significativo'],
        'Teste': row['Teste']
    })

df_efeitos = pd.DataFrame(todos_efeitos)

# Criar Forest Plot
plt.figure(figsize=(12, 8), dpi=600)

# Ordenar por tamanho de efeito
df_efeitos = df_efeitos.sort_values('Valor', ascending=True)

# Cores por significância
cores = df_efeitos['Significativo'].apply(lambda x: '#27ae60' if x else '#95a5a6')

# Plotar barras horizontais
y_pos = np.arange(len(df_efeitos))
bars = plt.barh(y_pos, df_efeitos['Valor'], color=cores, edgecolor='black', alpha=0.85)

# Adicionar valores
for i, (bar, valor) in enumerate(zip(bars, df_efeitos['Valor'])):
    plt.text(valor + 0.01, i, f'{valor:.3f}', va='center', fontsize=10, fontweight='bold')

# Linha de referência para efeito pequeno
plt.axvline(x=0.1, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Efeito Pequeno (referência)')

# Configurar gráfico
plt.yticks(y_pos, df_efeitos['Variavel'], fontsize=11)
plt.xlabel('Tamanho do Efeito', fontsize=12, fontweight='bold')
plt.ylabel('Variável', fontsize=12, fontweight='bold')
plt.title('Tamanhos de Efeito das Associações entre Procedimentos e Variáveis', 
          fontsize=14, fontweight='bold', pad=15)
plt.legend(loc='lower right', fontsize=10)
plt.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()

caminho_forest = os.path.join(DIR_GRAFICOS, 'forest_plot_tamanhos_efeito.png')
plt.savefig(caminho_forest, dpi=600, bbox_inches='tight')
plt.show()

logger.info(f"Forest plot salvo: {caminho_forest}")

In [ ]:
# TABELA CONSOLIDADA DE RESULTADOS INFERENCIAIS

display(Markdown("Tabela Consolidada de Resultados Inferenciais"))

tabela_publicacao = []

# Variáveis contínuas
for _, row in df_resultados_inferenciais.iterrows():
    tabela_publicacao.append({
        'Variavel': row['Variavel'],
        'Tipo_Variavel': 'Contínua',
        'Teste_Estatistico': row['Teste'],
        'Estatistica': float(row['Estatistica']),  # Manter como float
        'Graus_Liberdade': str(row['Gl']),
        'P_valor': row['P_valor_formatado'],  # Já formatado
        'Tamanho_Efeito': float(row['Tamanho_Efeito']),  # Manter como float
        'Medida_Efeito': row['Medida_Efeito'],
        'Significativo': 'Sim' if row['Significativo'] else 'Não'
    })

# Variáveis categóricas
for _, row in df_resultados_categoricos.iterrows():
    tabela_publicacao.append({
        'Variavel': row['Variavel'],
        'Tipo_Variavel': 'Categórica',
        'Teste_Estatistico': row['Teste'],
        'Estatistica': float(row['Estatistica']),  # Manter como float
        'Graus_Liberdade': str(row['Gl']),
        'P_valor': row['P_valor_formatado'],  # Já formatado
        'Tamanho_Efeito': float(row['Tamanho_Efeito']),  # Manter como float
        'Medida_Efeito': row['Medida_Efeito'],
        'Significativo': 'Sim' if row['Significativo'] else 'Não'
    })

df_tabela_publicacao = pd.DataFrame(tabela_publicacao)

# Exibir tabela formatada (formatação aplicada APENAS na exibição)
display(Markdown("**Tabela 3. Resultados dos Testes Inferenciais para Comparação entre Procedimentos Bariátricos**"))

display(df_tabela_publicacao.style
        .set_properties(**{'text-align': 'center'})
        .format({
            'Estatistica': '{:.2f}',
            'Tamanho_Efeito': '{:.3f}'
        })
        .background_gradient(subset=['Tamanho_Efeito'], cmap='YlOrRd'))

# Salvar tabela para submissão (com formatação aplicada)
df_export = df_tabela_publicacao.copy()
df_export['Estatistica'] = df_export['Estatistica'].apply(lambda x: f'{x:.2f}')
df_export['Tamanho_Efeito'] = df_export['Tamanho_Efeito'].apply(lambda x: f'{x:.3f}')

df_export.to_csv(
    os.path.join(DIR_TABELAS, "tabela_resultados_inferenciais_publicacao.csv"),
    index=False,
    encoding='utf-8',
    sep=';'
)

df_export.to_excel(
    os.path.join(DIR_TABELAS, "tabela_resultados_inferenciais_publicacao.xlsx"),
    index=False,
    engine='openpyxl'
)

logger.info("Tabela consolidada de resultados inferenciais salva")

In [ ]:
# Validação de consistência
validacoes = []

total_testes = len(df_tabela_publicacao)
validacoes.append({
    'Verificação': 'Total de testes inferenciais',
    'Valor': total_testes,
    'Status': 'OK' if total_testes > 0 else 'ERRO'
})

# Verificar se p-valores estão no formato correto 
pvalores_ok = all(
    (row['P_valor'].startswith('p-valor=') or row['P_valor'].startswith('p-valor<'))
    for _, row in df_tabela_publicacao.iterrows()
    if pd.notna(row['P_valor'])
)
validacoes.append({
    'Verificação': 'Formatação de p-valores',
    'Valor': pvalores_ok,
    'Status': 'OK' if pvalores_ok else 'ERRO'
})

df_validacao = pd.DataFrame(validacoes)
display(df_validacao.style.apply(
    lambda x: ['background-color: #d4edda' if v == 'OK' else 'background-color: #f8d7da' for v in x],
    subset=['Status']
))

In [ ]:
display(Markdown("Concluído"))

# Resumo das análises realizadas
resumo6 = {
    'Variáveis contínuas analisadas': len(df_resultados_inferenciais),
    'Variáveis categóricas analisadas': len(df_resultados_categoricos),
    'Testes de normalidade (Shapiro-Wilk)': len(df_normalidade),
    'Testes de homocedasticidade (Levene)': len(df_levene),
    'Testes post-hoc realizados': len(tabelas_posthoc),
    'Tamanhos de efeito calculados': len(df_tabela_publicacao),
    'P-valores formatados': pvalores_ok
}

display(pd.DataFrame(list(resumo6.items()), columns=['Métrica', 'Valor']).style.set_properties(**{'text-align': 'left'}))

# Resumo dos achados principais
display(Markdown("### Principais Achados Inferenciais"))

achados_sig = df_tabela_publicacao[df_tabela_publicacao['Significativo'] == 'Sim']
if len(achados_sig) > 0:
    for _, row in achados_sig.iterrows():
        display(Markdown(f"- **{row['Variavel']}**: {row['Teste_Estatistico']} ({row['Estatistica']}, {row['P_valor']}), {row['Medida_Efeito']}={row['Tamanho_Efeito']}"))
else:
    display(Markdown("- Nenhuma associação estatisticamente significativa identificada"))

logger.info("="*60)
logger.info("BLOCO 6 CONCLUÍDO COM SUCESSO")
logger.info("="*60)
logger.info(f"Testes inferenciais: {total_testes}")
logger.info(f"Tamanhos de efeito: {len(df_tabela_publicacao)}")
logger.info(f"P-valores formatados: {pvalores_ok}")
logger.info("="*60)

print("\nAnálise Inferencial e Comparativa - CONCLUÍDO")
print(f"Tabelas salvas em: {os.path.abspath(DIR_TABELAS)}")

In [ ]:
# ANÁLISE DE CUSTOS E EFICIÊNCIA

In [ ]:
CAMINHO_DATASET_LIMPO = os.path.join(DIR_DADOS, "dataset_limpo.parquet")
df = pd.read_parquet(CAMINHO_DATASET_LIMPO)

logger.info(f"Dataset carregado para análise de custos: {len(df):,} registros")

In [ ]:
# Taxas anuais de inflação IPCA
TAXAS_IPCA_ANUAL = {
    2016: 6.29, 2017: 2.95, 2018: 3.75, 2019: 4.31,
    2020: 4.52, 2021: 10.06, 2022: 5.79, 2023: 4.62,
    2024: 4.83, 2025: 4.26  
}

def calcular_fatores_cumulativos_ipca(taxas_anuais, ano_base=2024):
    """
    Converte taxas anuais de inflação IPCA em fatores cumulativos para ajuste de custos.
    Fator = 1.0 no ano base; anos anteriores: fator < 1; anos posteriores: fator > 1
    """
    fatores = {}
    fatores[ano_base] = 1.0
    
    # Calcular fatores para anos anteriores ao base
    fator_acumulado = 1.0
    for ano in range(ano_base - 1, min(taxas_anuais.keys()) - 1, -1):
        if ano in taxas_anuais:
            fator_acumulado = fator_acumulado / (1 + taxas_anuais[ano] / 100)
            fatores[ano] = round(fator_acumulado, 4)
    
    # Calcular fatores para anos posteriores ao base
    fator_acumulado = 1.0
    for ano in range(ano_base + 1, max(taxas_anuais.keys()) + 1):
        if ano in taxas_anuais:
            fator_acumulado = fator_acumulado * (1 + taxas_anuais[ano] / 100)
            fatores[ano] = round(fator_acumulado, 4)
    
    return fatores

# Calcular fatores cumulativos com base em 2024
IPCACORRECAO = calcular_fatores_cumulativos_ipca(TAXAS_IPCA_ANUAL, ano_base=2024)

# Validar cobertura de anos do dataset
anos_dataset = sorted(df['ANO_CMPT'].unique())
anos_sem_fator = [ano for ano in anos_dataset if ano not in IPCACORRECAO]

# Exibir fatores para conferência
display(Markdown("Fatores de Correção IPCA (Base: 2024)"))
fatores_df = pd.DataFrame({
    'Ano': sorted(IPCACORRECAO.keys()),
    'Taxa_IPCA_Anual (%)': [TAXAS_IPCA_ANUAL.get(ano, np.nan) for ano in sorted(IPCACORRECAO.keys())],
    'Fator_Cumulativo': [IPCACORRECAO[ano] for ano in sorted(IPCACORRECAO.keys())]
})
display(fatores_df.style.format({
    'Taxa_IPCA_Anual (%)': '{:.2f}%',
    'Fator_Cumulativo': '{:.4f}'
}))

# Componentes de custo disponíveis no SIH/SUS (conforme dicionário de variáveis)
COMPONENTES_CUSTO = {
    'VAL_SH': 'Serviços Hospitalares',
    'VAL_SP': 'Serviços Profissionais', 
    'VAL_SADT': 'Serviços Auxiliares de Diagnóstico e Terapia',
    'VAL_UTI': 'Unidade de Terapia Intensiva',
    'VAL_TOT': 'Custo Total da AIH'
}

logger.info(f"Fatores IPCA carregados para: {sorted(IPCACORRECAO.keys())}")

In [ ]:
def ajustar_custo_inflacao(df, coluna_custo='VAL_TOT', coluna_ano='ANO_CMPT', 
                           fatores_ipca=IPCACORRECAO, ano_base=2024,
                           aplicar_ajuste=True):
    """
    Ajusta valores de custo pela inflação (IPCA) para ano base específico.
    
    Parâmetros:
    -----------
    aplicar_ajuste : bool
        Se False, retorna cópia sem ajuste (para análise nominal)
    
    Retorna:
    --------
    DataFrame com coluna de custo ajustado adicionada
    """
    df_ajustado = df.copy()
    
    if not aplicar_ajuste:
        df_ajustado[f'{coluna_custo}_AJUSTADO'] = df_ajustado[coluna_custo]
        df_ajustado['ANO_BASE_CUSTO'] = 'Nominal'
        logger.info("Análise de custos em valores NOMINAIS (sem ajuste inflacionário)")
        return df_ajustado
    
    # Calcular fator de conversão para cada ano
    fator_base = fatores_ipca.get(ano_base, 1.0)
    
    df_ajustado[f'{coluna_custo}_AJUSTADO'] = df_ajustado.apply(
        lambda row: row[coluna_custo] * fatores_ipca.get(
            int(row[coluna_ano]), fator_base
        ) / fator_base,
        axis=1
    )
    df_ajustado['ANO_BASE_CUSTO'] = f'{ano_base}'
    
    logger.info(f"Custos ajustados para valores reais de {ano_base} (IPCA-IPEADATA)")
    return df_ajustado

def identificar_outliers_iqr(dados, coluna, multiplicador=1.5):
    """
    Identifica outliers usando o método IQR (Intervalo Interquartil).
    Limites: Q1 - 1.5×IQR (inferior) e Q3 + 1.5×IQR (superior)
    """
    q1 = dados[coluna].quantile(0.25)
    q3 = dados[coluna].quantile(0.75)
    iqr = q3 - q1
    
    limite_inferior = q1 - (multiplicador * iqr)
    limite_superior = q3 + (multiplicador * iqr)
    
    outliers_inferiores = dados[dados[coluna] < limite_inferior]
    outliers_superiores = dados[dados[coluna] > limite_superior]
    dados_sem_outliers = dados[(dados[coluna] >= limite_inferior) & 
                                (dados[coluna] <= limite_superior)]
    
    return {
        'q1': q1, 'q3': q3, 'iqr': iqr,
        'limite_inferior': limite_inferior,
        'limite_superior': limite_superior,
        'n_outliers_inferiores': len(outliers_inferiores),
        'n_outliers_superiores': len(outliers_superiores),
        'n_outliers_total': len(outliers_inferiores) + len(outliers_superiores),
        'percentual_outliers': (len(outliers_inferiores) + len(outliers_superiores)) / len(dados) * 100,
        'dados_com_outliers': dados,
        'dados_sem_outliers': dados_sem_outliers
    }

logger.info("Funções de ajuste inflacionário e outliers carregadas")

In [ ]:
display(Markdown("##Estratégia Analítica: Valores Nominais vs. Ajustados por Inflação"))

# Aplicar análise dual
df_nominal = ajustar_custo_inflacao(df, aplicar_ajuste=False)
df_ajustado = ajustar_custo_inflacao(df, aplicar_ajuste=True)

# Comparação resumida
comparacao_dual = pd.DataFrame({
    'Métrica': [
        'Custo Médio Nominal',
        'Custo Médio Ajustado (2024)',
        'Diferença Relativa (%)',
        'Mediana Nominal',
        'Mediana Ajustada (2024)',
        'IIQ Nominal',
        'IIQ Ajustado (2024)'
    ],
    'Valor': [
        f"R$ {df_nominal['VAL_TOT'].mean():,.2f}",
        f"R$ {df_ajustado['VAL_TOT_AJUSTADO'].mean():,.2f}",
        f"{((df_ajustado['VAL_TOT_AJUSTADO'].mean() / df_nominal['VAL_TOT'].mean()) - 1) * 100:.1f}%",
        f"R$ {df_nominal['VAL_TOT'].median():,.2f}",
        f"R$ {df_ajustado['VAL_TOT_AJUSTADO'].median():,.2f}",
        f"R$ {(df_nominal['VAL_TOT'].quantile(0.75) - df_nominal['VAL_TOT'].quantile(0.25)):,.2f}",
        f"R$ {(df_ajustado['VAL_TOT_AJUSTADO'].quantile(0.75) - df_ajustado['VAL_TOT_AJUSTADO'].quantile(0.25)):,.2f}"
    ]
})

display(Markdown("### Tabela 1. Comparação: Custos Nominais vs. Ajustados por IPCA"))
display(comparacao_dual.style.set_properties(**{'text-align': 'center'}))

# Salvar para documentação 
comparacao_dual.to_csv(
    os.path.join(DIR_DADOS, "comparacao_custos_nominal_ajustado.csv"),
    index=False,
    encoding='utf-8'
)

# Nota 
display(Markdown(""))

In [ ]:
display(Markdown("Estatísticas Descritivas de Custos por Procedimento"))
display(Markdown(f"**Período analisado:** {df['ANO_CMPT'].min():.0f} - {df['ANO_CMPT'].max():.0f}"))
display(Markdown(f"**Total de procedimentos:** {len(df):,}"))

# Calcular estatísticas para ambas as análises (nominal e ajustada)
# Foco na mediana e IIQ como medidas primárias
estatisticas_custo = []

for proc_codigo, proc_nome in CODIGOS_BARIATRICOS.items():
    df_proc_nom = df_nominal[df_nominal['PROC_REA'] == proc_codigo]['VAL_TOT'].dropna()
    df_proc_adj = df_ajustado[df_ajustado['PROC_REA'] == proc_codigo]['VAL_TOT_AJUSTADO'].dropna()
    
    if len(df_proc_nom) == 0:
        continue
    
    # Identificar outliers (usando dados nominais para definição)
    outliers_info = identificar_outliers_iqr(
        pd.DataFrame({'VAL_TOT': df_proc_nom}), 'VAL_TOT'
    )
    
    estatisticas_custo.append({
        'Procedimento': proc_nome,
        'Codigo': proc_codigo,
        'N_Total': len(df_proc_nom),
        'N_Outliers': outliers_info['n_outliers_total'],
        'Percentual_Outliers': outliers_info['percentual_outliers'],
        # Valores nominais (mediana e IIQ como medidas primárias)
        'Mediana_Nominal': df_proc_nom.median(),
        'IIQ_Nominal': df_proc_nom.quantile(0.75) - df_proc_nom.quantile(0.25),
        'Q1_Nominal': df_proc_nom.quantile(0.25),
        'Q3_Nominal': df_proc_nom.quantile(0.75),
        'Media_Nominal': df_proc_nom.mean(), 
        'DP_Nominal': df_proc_nom.std(),
        # Valores ajustados (mediana e IIQ como medidas primárias)
        'Mediana_Ajustada': df_proc_adj.median(),
        'IIQ_Ajustada': df_proc_adj.quantile(0.75) - df_proc_adj.quantile(0.25),
        'Q1_Ajustada': df_proc_adj.quantile(0.25),
        'Q3_Ajustada': df_proc_adj.quantile(0.75),
        'Media_Ajustada': df_proc_adj.mean(),
        'DP_Ajustada': df_proc_adj.std()
    })

df_custo_stats = pd.DataFrame(estatisticas_custo)

# Exibir tabela formatada 

display(Markdown("Estatísticas de Custo por Tipo de Procedimento"))
display(df_custo_stats[[
    'Procedimento', 'N_Total', 'Mediana_Ajustada', 'IIQ_Ajustada', 
    'Percentual_Outliers', 'Media_Ajustada', 'DP_Ajustada'
]].style.format({
    'N_Total': '{:,.0f}',
    'Mediana_Ajustada': 'R$ {:,.2f}',
    'IIQ_Ajustada': 'R$ {:,.2f}',
    'Percentual_Outliers': '{:.2f}%',
    'Media_Ajustada': 'R$ {:,.2f}',
    'DP_Ajustada': 'R$ {:,.2f}'
}).background_gradient(subset=['Mediana_Ajustada'], cmap='YlOrRd'))

# Salvar tabela completa com ambas as métricas
df_custo_stats.to_csv(
    os.path.join(DIR_TABELAS, "estatisticas_custo_dual.csv"),
    index=False,
    encoding='utf-8',
    sep=';'
)

logger.info("Estatísticas de custo dual calculadas e salvas")

In [ ]:
display(Markdown("Distribuição de Custos Ajustados por Procedimento"))

plt.figure(figsize=(14, 10), dpi=600)

# Preparar dados para violin plot
dados_violin = []
nomes_violin = []
cores_violin = []

for proc_codigo, proc_nome in CODIGOS_BARIATRICOS.items():
    dados = df_ajustado[df_ajustado['PROC_REA'] == proc_codigo]['VAL_TOT_AJUSTADO'].dropna()
    if len(dados) > 0:
        dados_violin.append(dados.values)
        nomes_violin.append(proc_nome)
        cores_violin.append(CORES_PROCEDIMENTOS.get(proc_codigo, '#3498db'))

# Criar violin plot
parts = plt.violinplot(dados_violin, positions=range(len(dados_violin)), 
                       showmeans=False, showmedians=True, showextrema=True,
                       widths=0.7)

# Colorir violinos com paleta acessível
for i, (pc, cor) in enumerate(zip(parts['bodies'], cores_violin)):
    pc.set_facecolor(cor)
    pc.set_alpha(0.6)
    pc.set_edgecolor('black')
    pc.set_linewidth(1.5)

# Personalizar medianas e extremos
for partname in ('cbars', 'cmins', 'cmaxes', 'cmedians'):
    vp = parts[partname]
    vp.set_edgecolor('black')
    vp.set_linewidth(2)

# Configurar gráfico
plt.xticks(range(len(nomes_violin)), nomes_violin, rotation=15, ha='right', fontsize=11)
plt.ylabel('Custo Total Ajustado (R$ de 2024)', fontsize=12, fontweight='bold')
plt.title(f'Distribuição de Custos por Tipo de Procedimento Bariátrico, Brasil, 2016-2025 (n={len(df):,}; valores ajustados IPCA)', 
          fontsize=14, fontweight='bold', pad=15)
plt.grid(axis='y', alpha=0.3, linestyle='--')

# Format eixo Y para moeda
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'R$ {x:,.0f}'))

# Adicionar estatísticas de forma legível
for i, dados in enumerate(dados_violin):
    n = len(dados)
    mediana = np.median(dados)
    iiq = np.percentile(dados, 75) - np.percentile(dados, 25)
    plt.text(i, plt.ylim()[1] * 0.95, f'N={n:,}\nMd=R$ {mediana:,.0f}\nIIQ=R$ {iiq:,.0f}', 
             ha='center', va='top', fontsize=9,
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', 
                      boxstyle='round,pad=0.3'))

plt.tight_layout()
caminho_violin = os.path.join(DIR_GRAFICOS, 'violinplot_custos_ajustados_2024.png')
plt.savefig(caminho_violin, dpi=600, bbox_inches='tight')
plt.show()

logger.info(f"Violin plot de custos ajustados salvo: {caminho_violin}")

In [ ]:
display(Markdown("Análise de Sensibilidade: Impacto dos Outliers"))

# Análise comparativa com e sem outliers (usando valores ajustados)
analise_sensibilidade = []

for proc_codigo, proc_nome in CODIGOS_BARIATRICOS.items():
    df_proc = df_ajustado[df_ajustado['PROC_REA'] == proc_codigo][['VAL_TOT_AJUSTADO', 'PROC_REA']].dropna()
    
    if len(df_proc) == 0:
        continue
    
    # Identificar outliers nos valores ajustados
    outliers_info = identificar_outliers_iqr(df_proc, 'VAL_TOT_AJUSTADO')
    
    # Estatísticas COM outliers
    media_com = df_proc['VAL_TOT_AJUSTADO'].mean()
    mediana_com = df_proc['VAL_TOT_AJUSTADO'].median()
    dp_com = df_proc['VAL_TOT_AJUSTADO'].std()
    
    # Estatísticas SEM outliers
    media_sem = outliers_info['dados_sem_outliers']['VAL_TOT_AJUSTADO'].mean()
    mediana_sem = outliers_info['dados_sem_outliers']['VAL_TOT_AJUSTADO'].median()
    dp_sem = outliers_info['dados_sem_outliers']['VAL_TOT_AJUSTADO'].std()
    
    # Calcular diferença percentual
    diff_media = ((media_sem - media_com) / media_com) * 100 if media_com != 0 else np.nan
    diff_mediana = ((mediana_sem - mediana_com) / mediana_com) * 100 if mediana_com != 0 else np.nan
    
    analise_sensibilidade.append({
        'Procedimento': proc_nome,
        'N_Total': len(df_proc),
        'N_Outliers': outliers_info['n_outliers_total'],
        'Percentual_Outliers': outliers_info['percentual_outliers'],
        'Media_Com_Outliers': media_com,
        'Media_Sem_Outliers': media_sem,
        'Diferenca_Media_Pct': diff_media,
        'Mediana_Com_Outliers': mediana_com,
        'Mediana_Sem_Outliers': mediana_sem,
        'Diferenca_Mediana_Pct': diff_mediana,
        'DP_Com': dp_com,
        'DP_Sem': dp_sem
    })

df_sensibilidade = pd.DataFrame(analise_sensibilidade)

display(Markdown("Análise de Sensibilidade dos Custos Ajustados (Com e Sem Outliers)"))
display(df_sensibilidade.style.format({
    'N_Total': '{:,.0f}',
    'N_Outliers': '{:,.0f}',
    'Percentual_Outliers': '{:.2f}%',
    'Media_Com_Outliers': 'R$ {:,.2f}',
    'Media_Sem_Outliers': 'R$ {:,.2f}',
    'Diferenca_Media_Pct': '{:+.2f}%',
    'Mediana_Com_Outliers': 'R$ {:,.2f}',
    'Mediana_Sem_Outliers': 'R$ {:,.2f}',
    'Diferenca_Mediana_Pct': '{:+.2f}%',
    'DP_Com': 'R$ {:,.2f}',
    'DP_Sem': 'R$ {:,.2f}'
}).background_gradient(subset=['Percentual_Outliers'], cmap='Reds'))

# Salvar tabela
df_sensibilidade.to_csv(
    os.path.join(DIR_TABELAS, "analise_sensibilidade_outliers_ajustado.csv"),
    index=False,
    encoding='utf-8',
    sep=';'
)

# Nota
display(Markdown(""))

logger.info("Análise de sensibilidade concluída")

In [ ]:
display(Markdown("Custo por Dia de Internação (Valores Ajustados)"))

# Calcular custo por dia
# Variáveis conforme dicionário: VAL_TOT (custo total), DIAS_PERM (dias de permanência)
df_custo_dia = df_ajustado[
    (df_ajustado['VAL_TOT_AJUSTADO'] > 0) & 
    (df_ajustado['DIAS_PERM'] > 0) & 
    (df_ajustado['DIAS_PERM'] <= 90) 
].copy()
df_custo_dia['CUSTO_POR_DIA_AJUSTADO'] = (
    df_custo_dia['VAL_TOT_AJUSTADO'] / df_custo_dia['DIAS_PERM']
)

# Estatísticas por procedimento (mediana e IIQ como medidas primárias)
custo_dia_stats = []

for proc_codigo, proc_nome in CODIGOS_BARIATRICOS.items():
    dados = df_custo_dia[
        df_custo_dia['PROC_REA'] == proc_codigo
    ]['CUSTO_POR_DIA_AJUSTADO'].dropna()
    
    if len(dados) == 0:
        continue
    
    custo_dia_stats.append({
        'Procedimento': proc_nome,
        'N': len(dados),
        'Mediana': dados.median(),
        'IIQ': dados.quantile(0.75) - dados.quantile(0.25),
        'Q1': dados.quantile(0.25),
        'Q3': dados.quantile(0.75),
        'Media': dados.mean(),
        'Desvio_Padrao': dados.std()
    })

df_custo_dia_stats = pd.DataFrame(custo_dia_stats)

display(Markdown("### Tabela 4. Custo por Dia de Internação por Procedimento (R$ de 2024)"))
display(df_custo_dia_stats.style.format({
    'N': '{:,.0f}',
    'Mediana': 'R$ {:,.2f}',
    'IIQ': 'R$ {:,.2f}',
    'Q1': 'R$ {:,.2f}',
    'Q3': 'R$ {:,.2f}',
    'Media': 'R$ {:,.2f}',
    'Desvio_Padrao': 'R$ {:,.2f}'
}).background_gradient(subset=['Mediana'], cmap='YlOrRd'))

# Salvar tabela
df_custo_dia_stats.to_csv(
    os.path.join(DIR_TABELAS, "custo_por_dia_ajustado.csv"),
    index=False,
    encoding='utf-8',
    sep=';'
)

# Visualização
plt.figure(figsize=(12, 8), dpi=600)

barras = plt.bar(
    df_custo_dia_stats['Procedimento'],
    df_custo_dia_stats['Mediana'],
    yerr=df_custo_dia_stats['Desvio_Padrao'],
    capsize=5,
    color=[CORES_PROCEDIMENTOS.get(cod, '#3498db') 
           for cod in df_custo_dia_stats['Procedimento'].map(
               {v: k for k, v in CODIGOS_BARIATRICOS.items()}
           )],
    edgecolor='black',
    alpha=0.85
)

# Adicionar valores com formatação legível
for bar, mediana in zip(barras, df_custo_dia_stats['Mediana']):
    plt.text(bar.get_x() + bar.get_width()/2, mediana + 100,
             f'R$ {mediana:,.0f}', ha='center', va='bottom',
             fontsize=10, fontweight='bold')

plt.ylabel('Custo por Dia Ajustado (R$ de 2024)', fontsize=12, fontweight='bold')
plt.title(f'Custo Médio por Dia de Internação por Procedimento, Brasil, 2016-2025 (n={len(df_custo_dia):,})',
          fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=15, ha='right')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'R$ {x:,.0f}'))
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()

caminho_custo_dia = os.path.join(DIR_GRAFICOS, 'custo_por_dia_ajustado.png')
plt.savefig(caminho_custo_dia, dpi=600, bbox_inches='tight')
plt.show()

logger.info(f"Gráfico de custo por dia ajustado salvo: {caminho_custo_dia}")

In [ ]:
# EVOLUÇÃO TEMPORAL DE CUSTOS


display(Markdown("Evolução Temporal: Custos Nominais vs. Ajustados por IPCA"))

# Agrupar por ano e procedimento com agregação correta
custo_temporal = df_ajustado.groupby(['ANO_CMPT', 'PROC_REA']).agg(
    Media_Nominal=('VAL_TOT', 'mean'),
    Media_Ajustada=('VAL_TOT_AJUSTADO', 'mean'),
    N=('VAL_TOT_AJUSTADO', 'count')
).reset_index()

# Renomear colunas de grupo
custo_temporal = custo_temporal.rename(columns={
    'ANO_CMPT': 'ANO',
    'PROC_REA': 'PROC_CODIGO'
})

# Adicionar nome do procedimento
custo_temporal['Procedimento'] = custo_temporal['PROC_CODIGO'].map(CODIGOS_BARIATRICOS)

# Verificar estrutura
print(f"Colunas geradas: {custo_temporal.columns.tolist()}")
print(f"Total de registros: {len(custo_temporal)}")

# Visualização comparativa
fig, axes = plt.subplots(1, 2, figsize=(18, 7), dpi=600)

# Custos Nominais
for proc_codigo, proc_nome in CODIGOS_BARIATRICOS.items():
    dados_proc = custo_temporal[custo_temporal['PROC_CODIGO'] == proc_codigo]
    if len(dados_proc) > 0:
        axes[0].plot(
            dados_proc['ANO'],
            dados_proc['Media_Nominal'],
            marker='o',
            markersize=6,
            linewidth=2.0,
            label=proc_nome,
            color=CORES_PROCEDIMENTOS.get(proc_codigo, '#3498db'),
            alpha=0.85
        )

axes[0].set_xlabel('Ano', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Custo Médio Nominal (R$)', fontsize=11, fontweight='bold')
axes[0].set_title('Evolução de Custos: Valores Nominais', fontsize=13, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].grid(True, alpha=0.3, linestyle='--')
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'R$ {x:,.0f}'))

# Custos Ajustados
for proc_codigo, proc_nome in CODIGOS_BARIATRICOS.items():
    dados_proc = custo_temporal[custo_temporal['PROC_CODIGO'] == proc_codigo]
    if len(dados_proc) > 0:
        axes[1].plot(
            dados_proc['ANO'],
            dados_proc['Media_Ajustada'],
            marker='o',
            markersize=6,
            linewidth=2.0,
            label=proc_nome,
            color=CORES_PROCEDIMENTOS.get(proc_codigo, '#3498db'),
            alpha=0.85
        )

axes[1].set_xlabel('Ano', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Custo Médio Ajustado (R$ de 2024)', fontsize=11, fontweight='bold')
axes[1].set_title('Evolução de Custos: Valores Ajustados por IPCA', fontsize=13, fontweight='bold')
axes[1].legend(loc='upper left', fontsize=9)
axes[1].grid(True, alpha=0.3, linestyle='--')
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'R$ {x:,.0f}'))

plt.suptitle(f'Comparação Temporal de Custos por Procedimento, Brasil, 2016-2025 (n={len(df):,})', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

caminho_evolucao = os.path.join(DIR_GRAFICOS, 'evolucao_custos_nominal_vs_ajustado.png')
plt.savefig(caminho_evolucao, dpi=600, bbox_inches='tight')
plt.show()

# Calcular taxas de crescimento real
display(Markdown("### Taxas de Crescimento Real de Custos (Descontada Inflação)"))

for proc_codigo, proc_nome in CODIGOS_BARIATRICOS.items():
    dados_proc = custo_temporal[custo_temporal['PROC_CODIGO'] == proc_codigo]
    if len(dados_proc) >= 2:
        custo_inicial = dados_proc['Media_Ajustada'].iloc[0]
        custo_final = dados_proc['Media_Ajustada'].iloc[-1]
        anos = len(dados_proc) - 1
        
        if pd.notna(custo_inicial) and pd.notna(custo_final) and custo_inicial > 0 and anos > 0:
            taxa_crescimento = ((custo_final / custo_inicial) ** (1/anos) - 1) * 100
            display(Markdown(f"- **{proc_nome}:** {taxa_crescimento:+.2f}% ao ano (real, ajustado IPCA)"))

logger.info("Evolução temporal comparativa concluída")

In [ ]:
display(Markdown("Comparação Estatística de Custos entre Procedimentos"))

# Preparar grupos para teste
grupos_custo = []
nomes_grupos = []

for proc_codigo, proc_nome in CODIGOS_BARIATRICOS.items():
    dados = df_ajustado[df_ajustado['PROC_REA'] == proc_codigo]['VAL_TOT_AJUSTADO'].dropna()
    if len(dados) >= 5:
        grupos_custo.append(dados.values)
        nomes_grupos.append(proc_nome)

if len(grupos_custo) >= 2:
    # Teste de normalidade (Shapiro-Wilk)
    display(Markdown("###Teste de Normalidade (Shapiro-Wilk)"))
    
    resultados_normalidade = []
    for i, (grupo, nome) in enumerate(zip(grupos_custo, nomes_grupos)):
        if 3 < len(grupo) <= 5000:
            stat, p_valor = shapiro(grupo)
            resultados_normalidade.append({
                'Procedimento': nome,
                'N': len(grupo),
                'Estatistica_W': stat,
                'P_valor': p_valor,
                'Normal': p_valor > 0.05
            })
    
    df_normalidade_custo = pd.DataFrame(resultados_normalidade)
    display(df_normalidade_custo.style.format({
        'N': '{:,.0f}',
        'Estatistica_W': '{:.4f}',
        'P_valor': '{:.4f}'
    }).applymap(lambda x: 'background-color: #d4edda' if x == True 
                else 'background-color: #f8d7da' if x == False 
                else '', subset=['Normal']))
    
    # Decisão: Paramétrico ou Não-Paramétrico
    todos_normais = df_normalidade_custo['Normal'].all()
    
    if todos_normais:
        display(Markdown("**Decisão:** Todos os grupos seguem distribuição normal → ANOVA"))
        stat_teste, p_teste = f_oneway(*grupos_custo)
        nome_teste = 'ANOVA One-Way'
    else:
        display(Markdown("**Decisão:** Pelo menos um grupo não é normal → Kruskal-Wallis"))
        stat_teste, p_teste = kruskal(*grupos_custo)
        nome_teste = 'Kruskal-Wallis'
    
    # Resultado do teste
    display(Markdown(f"### 6.2 Resultado do Teste ({nome_teste})"))
    display(Markdown(f"""
| Estatística | Valor |
|-------------|-------|
| **Teste** | {nome_teste} |
| **Estatística** | {stat_teste:.4f} |
| **Graus de liberdade** | {len(grupos_custo) - 1} |
| **p-valor** | {p_teste:.4f} |
| **Significância (α=0,05)** | {'SIM' if p_teste < 0.05 else 'NÃO'} |
"""))
    
    # Formatar p-valor
    def formatar_pvalor(p):
        if pd.isna(p):
            return 'p-valor=NA'
        elif p < 0.001:
            return 'p-valor<0,001'
        else:
            return f'p-valor={p:.3f}'.replace('.', ',')
    
    display(Markdown(f"**Formato:** {formatar_pvalor(p_teste)}"))
    
    # Salvar resultados
    resultados_teste_custo = {
        'Teste': nome_teste,
        'Estatistica': stat_teste,
        'Gl': len(grupos_custo) - 1,
        'P_valor': p_teste,
        'P_valor_formatado': formatar_pvalor(p_teste),
        'Significativo': p_teste < 0.05,
        'Tipo_Valor': 'Ajustado_IPCA_2024'
    }
    
    pd.DataFrame([resultados_teste_custo]).to_csv(
        os.path.join(DIR_TABELAS, "teste_comparacao_custos_ajustado.csv"),
        index=False,
        encoding='utf-8',
        sep=';'
    )
    
    logger.info(f"Teste de comparação de custos ajustados concluído: {nome_teste}")

In [ ]:
display(Markdown("Limitações IPCA"))

# Tabela de limitações específicas
limitacoes_ipca = pd.DataFrame()

display(limitacoes_ipca.style.set_properties(**{'text-align': 'left', 'vertical-align': 'top'}))

# Salvar documentação de limitações
limitacoes_ipca.to_csv(
    os.path.join(DIR_DADOS, "limitacoes_ipca_analise_custos.csv"),
    index=False,
    encoding='utf-8',
    sep=';'
)

# Nota
display(Markdown())

logger.info("")

In [ ]:
display(Markdown("Concluído"))

# Resumo das análises realizadas
resumo_bloco7 = {
    'Total de procedimentos analisados': len(df),
    'Procedimentos comparados': len(CODIGOS_BARIATRICOS),
    'Período': f"{df['ANO_CMPT'].min():.0f} - {df['ANO_CMPT'].max():.0f}",
    'Ano base do ajuste IPCA': '2024',
    'Custo mediano nominal': f"R$ {df_nominal['VAL_TOT'].median():,.2f}",
    'Custo mediano ajustado (2024)': f"R$ {df_ajustado['VAL_TOT_AJUSTADO'].median():,.2f}",
    'Percentual médio de outliers': f"{df_custo_stats['Percentual_Outliers'].mean():.2f}%",
    'Teste estatístico aplicado': nome_teste if len(grupos_custo) >= 2 else 'N/A',
    'Tabelas exportadas': 6,
    'Gráficos exportados': 3
}

display(pd.DataFrame(list(resumo_bloco7.items()), columns=['Métrica', 'Valor']).style.set_properties(**{'text-align': 'left'}))

# Validação de consistência
display(Markdown("### Validação de Consistência Numérica"))

validacoes = []

# Verificar se mediana ajustada é reportada para todas as variáveis de custo
mediana_ajustada_reportada = all(col in df_custo_stats.columns for col in ['Mediana_Ajustada', 'Q1_Ajustada', 'Q3_Ajustada', 'IIQ_Ajustada'])
validacoes.append({
    'Verificação': 'Mediana e IIQ ajustados reportados',
    'Status': 'OK' if mediana_ajustada_reportada else 'ERRO'
})

# Verificar se análise de sensibilidade foi realizada
sensibilidade_realizada = len(df_sensibilidade) > 0
validacoes.append({
    'Verificação': 'Análise de sensibilidade com outliers',
    'Status': 'OK' if sensibilidade_realizada else 'ERRO'
})

# Verificar se IPCA 2025 está incluído
ipca_2025_incluido = 2025 in IPCACORRECAO
validacoes.append({
    'Verificação': 'IPCA 2025 incluído no dicionário',
    'Status': 'OK' if ipca_2025_incluido else 'ERRO'
})

# Verificar se limitações do IPCA foram documentadas
limitacoes_documentadas = os.path.exists(
    os.path.join(DIR_DADOS, "limitacoes_ipca_analise_custos.csv")
)
validacoes.append({
    'Verificação': 'Limitações IPCA documentadas',
    'Status': 'OK' if limitacoes_documentadas else 'ERRO'
})

# Verificar se análise dual foi aplicada
analise_dual_aplicada = 'VAL_TOT_AJUSTADO' in df_ajustado.columns and 'ANO_BASE_CUSTO' in df_ajustado.columns
validacoes.append({
    'Verificação': 'Análise dual (nominal + ajustada) aplicada',
    'Status': 'OK' if analise_dual_aplicada else 'ERRO'
})

df_validacao_custo = pd.DataFrame(validacoes)
display(df_validacao_custo.style.apply(
    lambda x: ['background-color: #d4edda' if v == 'OK' 
               else 'background-color: #f8d7da' for v in x],
    subset=['Status']
))

# Exportar todas as tabelas de custo em Excel consolidado
with pd.ExcelWriter(os.path.join(DIR_TABELAS, "analise_custos_completa.xlsx"), 
                    engine='openpyxl') as writer:
    df_custo_stats.to_excel(writer, sheet_name='Estatisticas_Procedimento', index=False)
    df_sensibilidade.to_excel(writer, sheet_name='Sensibilidade_Outliers', index=False)
    df_custo_dia_stats.to_excel(writer, sheet_name='Custo_por_Dia', index=False)
    custo_temporal.to_excel(writer, sheet_name='Evolucao_Temporal', index=False)
    comparacao_dual.to_excel(writer, sheet_name='Comparacao_Nominal_Ajustado', index=False)

logger.info("Tabelas de custos consolidadas em Excel")

logger.info("="*60)
logger.info("BLOCO 7 CONCLUÍDO COM SUCESSO")
logger.info("="*60)
logger.info(f"Custo mediano ajustado: R$ {df_ajustado['VAL_TOT_AJUSTADO'].median():,.2f}")
logger.info(f"Outliers: {df_custo_stats['Percentual_Outliers'].mean():.2f}%")
logger.info(f"Teste: {nome_teste if len(grupos_custo) >= 2 else 'N/A'}")
logger.info(f"IPCA 2025 incluído: {ipca_2025_incluido}")
logger.info("="*60)

print("\nAnálise de Custos e Eficiência - CONCLUÍDO")
print(f"Tabelas salvas em: {os.path.abspath(DIR_TABELAS)}")
print(f"Gráficos salvos em: {os.path.abspath(DIR_GRAFICOS)}")
print(f"Documentação de limitações: {os.path.abspath(DIR_DADOS)}")

In [ ]:
# ANÁLISE DE CUSTOS ESTRATIFICADA POR PROCEDIMENTO


display(Markdown("## Análise de Custos Estratificada por Procedimento"))

resultados_custos = []

for cod_proc, nome_proc in CODIGOS_BARIATRICOS.items():
    df_proc = df[(df['PROC_REA'] == cod_proc) & (df['VAL_TOT'].notna())].copy()
    if len(df_proc) == 0:
        continue
    
    custos = df_proc['VAL_TOT']
    n = len(custos)
    
    # Estatísticas descritivas
    stats = {
        'Procedimento': NOMES_CURTOS.get(cod_proc, nome_proc),
        'Codigo': cod_proc,
        'N': n,
        'Media': custos.mean(),
        'Mediana': custos.median(),
        'DP': custos.std(),
        'Q1': custos.quantile(0.25),
        'Q3': custos.quantile(0.75),
        'IIQ': custos.quantile(0.75) - custos.quantile(0.25),
        'Minimo': custos.min(),
        'Maximo': custos.max(),
        'CV_Pct': (custos.std() / custos.mean() * 100) if custos.mean() > 0 else np.nan
    }
    
    # Teste de normalidade
    from scipy.stats import shapiro, kstest
    if n <= 5000:
        stat_norm, p_norm = shapiro(custos.sample(min(n, 5000), random_state=42))
        stats['Teste_Normalidade'] = 'Shapiro-Wilk'
    else:
        stat_norm, p_norm = kstest(custos, 'norm', args=(custos.mean(), custos.std()))
        stats['Teste_Normalidade'] = 'Kolmogorov-Smirnov'
    
    stats['P_valor_Normalidade'] = p_norm
    stats['Normal'] = p_norm > 0.05
    resultados_custos.append(stats)
    
    # Gráfico: Violin + Q-Q Plot
    from scipy.stats import probplot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), dpi=600)
    
    # Violin plot
    vp = ax1.violinplot([custos.values], positions=[0], showmeans=False, 
                       showmedians=True, showextrema=True, widths=0.8)
    for pc in vp['bodies']:
        pc.set_facecolor(CORES_PROCEDIMENTOS.get(cod_proc, '#3498db'))
        pc.set_alpha(0.7)
        pc.set_edgecolor('black')
    for part in ['cmins', 'cmaxes', 'cmedians', 'cbars']:
        if part in vp:
            vp[part].set_edgecolor('black')
            vp[part].set_linewidth(1.5)
    
    ax1.set_xticks([0])
    ax1.set_xticklabels([NOMES_CURTOS.get(cod_proc, nome_proc)])
    ax1.set_ylabel('Custo Total (R$)', fontsize=11, fontweight='bold')
    ax1.set_title('Distribuição de Custos', fontsize=12, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3, linestyle='--')
    ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'R$ {x:,.0f}'))
    
    # Q-Q Plot
    probplot(custos.sample(min(n, 1000), random_state=42), dist="norm", plot=ax2)
    ax2.set_title('Q-Q Plot (amostra)', fontsize=12, fontweight='bold')
    ax2.grid(alpha=0.3, linestyle='--')
    
    plt.suptitle(f'{NOMES_CURTOS.get(cod_proc, nome_proc)} | '
                f'Mediana: R$ {custos.median():,.2f} | IIQ: R$ {stats["IIQ"]:,.2f} | '
                f'CV: {stats["CV_Pct"]:.1f}% | Normal: {"Sim" if stats["Normal"] else "Não"}',
                fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    nome_arq = f"custos_{NOMES_CURTOS.get(cod_proc, cod_proc).replace(' ', '_')}.png"
    plt.savefig(os.path.join(DIR_GRAFICOS, nome_arq), dpi=600, bbox_inches='tight')
    plt.close()
    
    # Gráfico: Evolução temporal do custo médio
    custos_anuais = df_proc.groupby('ANO_CMPT')['VAL_TOT'].agg(['mean', 'count']).reset_index()
    if len(custos_anuais) >= 3:
        plt.figure(figsize=(12, 6), dpi=600)
        plt.plot(custos_anuais['ANO_CMPT'], custos_anuais['mean'], 
                marker='o', linewidth=2.5, color=CORES_PROCEDIMENTOS.get(cod_proc, '#3498db'))
        plt.fill_between(custos_anuais['ANO_CMPT'], 
                       custos_anuais['mean'] - custos_anuais['mean'].std(),
                       custos_anuais['mean'] + custos_anuais['mean'].std(),
                       alpha=0.2, color=CORES_PROCEDIMENTOS.get(cod_proc, '#3498db'))
        plt.xlabel('Ano', fontsize=11, fontweight='bold')
        plt.ylabel('Custo Médio (R$)', fontsize=11, fontweight='bold')
        plt.title(f'Evolução do Custo Médio: {NOMES_CURTOS.get(cod_proc, nome_proc)}', 
                 fontsize=12, fontweight='bold')
        plt.grid(alpha=0.3, linestyle='--')
        plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'R$ {x:,.0f}'))
        plt.tight_layout()
        
        nome_arq2 = f"custos_temporal_{NOMES_CURTOS.get(cod_proc, cod_proc).replace(' ', '_')}.png"
        plt.savefig(os.path.join(DIR_GRAFICOS, nome_arq2), dpi=600, bbox_inches='tight')
        plt.close()

# Tabela consolidada
df_resultados_custos = pd.DataFrame(resultados_custos)
if not df_resultados_custos.empty:
    df_resultados_custos.to_csv(
        os.path.join(DIR_TABELAS, "analise_custos_por_procedimento.csv"),
        index=False, encoding='utf-8', sep=';'
    )
    display(Markdown("### Resumo da Análise de Custos por Procedimento"))
    display(df_resultados_custos.style.format({
        'N': '{:,.0f}',
        'Media': 'R$ {:,.2f}',
        'Mediana': 'R$ {:,.2f}',
        'DP': 'R$ {:,.2f}',
        'IIQ': 'R$ {:,.2f}',
        'CV_Pct': '{:.1f}%',
        'P_valor_Normalidade': '{:.4f}'
    }).applymap(lambda x: 'background-color: #d4edda' if x == True 
                else 'background-color: #f8d7da' if x == False 
                else '', subset=['Normal']))

logger.info(f"Análise de custos estratificada concluída: {len(resultados_custos)} procedimentos")

In [ ]:
# EXPORTAÇÃO, VALIDAÇÃO E DOCUMENTAÇÃO FINAL

In [ ]:
# Carregar dataset final
CAMINHO_DATASET_LIMPO = os.path.join(DIR_DADOS, "dataset_limpo.parquet")
df = pd.read_parquet(CAMINHO_DATASET_LIMPO)

In [ ]:
# Carregar resultados dos blocos anteriores
CAMINHO_RESULTADOS = {
    'descritiva': os.path.join(DIR_TABELAS, "estatisticas_custo_dual.csv"),
    'temporal': os.path.join(DIR_TABELAS, "modelo_regressao_segmentada.csv"),
    'espacial': os.path.join(DIR_TABELAS, "taxas_procedimentos_censo2022.csv"),
    'inferencial': os.path.join(DIR_TABELAS, "resultados_inferenciais_continuas.csv"),
    'gini': os.path.join(DIR_DADOS, "indice_gini_resultados.csv")
}

# Carregar dados disponíveis
resultados = {}
for nome, caminho in CAMINHO_RESULTADOS.items():
    if os.path.exists(caminho):
        resultados[nome] = pd.read_csv(caminho, sep=';', encoding='utf-8')
        logger.info(f"Resultados {nome} carregados: {len(resultados[nome])} registros")

logger.info(f"Dataset final: {len(df):,} registros")

In [ ]:
def validar_consistencia_numerica(df, resultados):
    """Valida consistência entre diferentes fontes de resultados"""
    validacoes = []
    
    # Total de procedimentos
    total_dataset = len(df)
    validacoes.append({
        'Verificação': 'Total de procedimentos no dataset',
        'Valor': total_dataset,
        'Status': 'OK' if total_dataset > 0 else 'ERRO'
    })
    
    # Verificar períodos
    periodo_min = df['ANO_CMPT'].min()
    periodo_max = df['ANO_CMPT'].max()
    validacoes.append({
        'Verificação': 'Período do estudo',
        'Valor': f"{periodo_min:.0f}-{periodo_max:.0f}",
        'Status': 'OK' if periodo_max >= periodo_min else 'ERRO'
    })
    
    # Verificar procedimentos
    n_procedimentos = df['PROC_REA'].nunique() if 'PROC_REA' in df.columns else 0
    validacoes.append({
        'Verificação': 'Tipos de procedimento',
        'Valor': n_procedimentos,
        'Status': 'OK' if n_procedimentos >= 1 else 'ERRO'
    })
    
    # Verificar UFs
    n_ufs = df['UF_SIGLA'].nunique() if 'UF_SIGLA' in df.columns else 0
    validacoes.append({
        'Verificação': 'UFs cobertas',
        'Valor': n_ufs,
        'Status': 'OK' if n_ufs >= 1 else 'ERRO'
    })
    
    # Verificar regiões
    n_regioes = df['REGIAO'].nunique() if 'REGIAO' in df.columns else 0
    validacoes.append({
        'Verificação': 'Regiões cobertas',
        'Valor': n_regioes,
        'Status': 'OK' if n_regioes >= 1 else 'ERRO'
    })
    
    return pd.DataFrame(validacoes)

df_validacao = validar_consistencia_numerica(df, resultados)

display(Markdown("## Validação de Consistência Numérica"))
display(df_validacao.style.apply(
    lambda x: ['background-color: #d4edda' if v == 'OK' 
               else 'background-color: #f8d7da' for v in x],
    subset=['Status']
))

# Salvar validação
df_validacao.to_csv(
    os.path.join(DIR_DADOS, "validacao_consistencia_final.csv"),
    index=False,
    encoding='utf-8'
)

In [ ]:
def formatar_tabela(df, nome_tabela, caminho_saida):
    # Criar workbook Excel
    with pd.ExcelWriter(caminho_saida, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=nome_tabela[:31], index=False)
        
        # Formatar planilha
        ws = writer.sheets[nome_tabela[:31]]
        
        # Formatar cabeçalhos
        for cell in ws[1]:
            cell.font = openpyxl.styles.Font(bold=True, size=11)
            cell.fill = openpyxl.styles.PatternFill(start_color='4472C4', 
                                                     end_color='4472C4', 
                                                     fill_type='solid')
            cell.alignment = openpyxl.styles.Alignment(horizontal='center', 
                                                        vertical='center')
        
        # Ajustar largura das colunas
        for column in ws.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                if cell.value:
                    max_length = max(max_length, len(str(cell.value)))
            ws.column_dimensions[column_letter].width = min(max_length + 2, 50)
    
    logger.info(f"Tabela exportada: {caminho_saida}")
    return caminho_saida

# Exportar tabelas principais
tabelas_exportar = {
    'caracteristicas_gerais': os.path.join(DIR_TABELAS, "tabela_01_caracteristicas_gerais.xlsx"),
    'distribuicao_procedimentos': os.path.join(DIR_TABELAS, "tabela_02_distribuicao_procedimentos.xlsx"),
    'analise_temporal': os.path.join(DIR_TABELAS, "tabela_03_analise_temporal.xlsx"),
    'distribuicao_regional': os.path.join(DIR_TABELAS, "tabela_04_distribuicao_regional.xlsx"),
    'custos_procedimentos': os.path.join(DIR_TABELAS, "tabela_05_custos_procedimentos.xlsx"),
    'testes_inferenciais': os.path.join(DIR_TABELAS, "tabela_06_testes_inferenciais.xlsx")
}

# Gerar e exportar cada tabela
# Tabela 1: Características gerais
tabela_1 = pd.DataFrame({
    'Variável': ['Total de procedimentos', 'Período do estudo', 'UFs cobertas', 'Regiões cobertas'],
    'Valor': [
        f"{len(df):,}",
        f"{df['ANO_CMPT'].min():.0f}-{df['ANO_CMPT'].max():.0f}",
        df['UF_SIGLA'].nunique() if 'UF_SIGLA' in df.columns else 'N/A',
        df['REGIAO'].nunique() if 'REGIAO' in df.columns else 'N/A'
    ]
})
formatar_tabela(tabela_1, 'Caracteristicas_Gerais', tabelas_exportar['caracteristicas_gerais'])

# Tabela 2: Distribuição de procedimentos
if 'NOME_PROCEDIMENTO' in df.columns:
    tabela_2 = df['NOME_PROCEDIMENTO'].value_counts().reset_index()
    tabela_2.columns = ['Procedimento', 'N']
    tabela_2['Percentual (%)'] = (tabela_2['N'] / tabela_2['N'].sum() * 100).round(2)
    formatar_tabela(tabela_2, 'Distribuicao_Procedimentos', tabelas_exportar['distribuicao_procedimentos'])

# Tabela 3: Análise temporal
if 'temporal' in resultados:
    formatar_tabela(resultados['temporal'], 'Analise_Temporal', tabelas_exportar['analise_temporal'])

# Tabela 4: Distribuição regional
if 'REGIAO' in df.columns:
    tabela_4 = df['REGIAO'].value_counts().reset_index()
    tabela_4.columns = ['Região', 'N']
    tabela_4['Percentual (%)'] = (tabela_4['N'] / tabela_4['N'].sum() * 100).round(2)
    formatar_tabela(tabela_4, 'Distribuicao_Regional', tabelas_exportar['distribuicao_regional'])

# Tabela 5: Custos
if 'descritiva' in resultados:
    formatar_tabela(resultados['descritiva'], 'Custos_Procedimentos', tabelas_exportar['custos_procedimentos'])

# Tabela 6: Testes inferenciais
if 'inferencial' in resultados:
    formatar_tabela(resultados['inferencial'], 'Testes_Inferenciais', tabelas_exportar['testes_inferenciais'])

display(Markdown(f"**{len(tabelas_exportar)} tabelas exportadas**"))

In [ ]:
def validar_resolucao_figuras(dir_graficos, dpi_minimo=500):
    """Valida se todas as figuras atendem ao requisito de resolução"""
    validacoes = []
    
    if not os.path.exists(dir_graficos):
        return pd.DataFrame(validacoes)
    
    for arquivo in os.listdir(dir_graficos):
        if arquivo.endswith('.png'):
            caminho = os.path.join(dir_graficos, arquivo)
            try:
                from PIL import Image
                img = Image.open(caminho)
                dpi = img.info.get('dpi', (72, 72))
                
                validacoes.append({
                    'Arquivo': arquivo,
                    'DPI': dpi[0] if isinstance(dpi, tuple) else dpi,
                    'Status': 'OK' if dpi[0] >= dpi_minimo else 'ABAIXO_DO_PADRAO'
                })
            except Exception as e:
                validacoes.append({
                    'Arquivo': arquivo,
                    'DPI': 'N/A',
                    'Status': f'ERRO: {str(e)}'
                })
    
    return pd.DataFrame(validacoes)

# Validar figuras
df_validacao_figuras = validar_resolucao_figuras(DIR_GRAFICOS)

if len(df_validacao_figuras) > 0:
    display(Markdown("## Validação de Resolução de Figuras"))
    display(df_validacao_figuras.style.apply(
        lambda x: ['background-color: #d4edda' if v == 'OK' 
                   else 'background-color: #fff3cd' if v == 'ABAIXO_DO_PADRAO'
                   else 'background-color: #f8d7da' for v in x],
        subset=['Status']
    ))
    
    # Salvar validação
    df_validacao_figuras.to_csv(
        os.path.join(DIR_DADOS, "validacao_resolucao_figuras.csv"),
        index=False,
        encoding='utf-8'
    )

# Lista de figuras para submissão
figuras_submissao = []
if os.path.exists(DIR_GRAFICOS):
    for arquivo in os.listdir(DIR_GRAFICOS):
        if arquivo.endswith('.png'):
            figuras_submissao.append({
                'Arquivo': arquivo,
                'Caminho': os.path.join(DIR_GRAFICOS, arquivo),
                'Formato': 'PNG',
                'DPI': 600,
                'Uso_Sugerido': 'Submissão'
            })

df_figuras = pd.DataFrame(figuras_submissao)
if len(df_figuras) > 0:
    df_figuras.to_csv(
        os.path.join(DIR_RESULTADOS, "lista_figuras_submissao.csv"),
        index=False,
        encoding='utf-8'
    )
    display(Markdown(f"**{len(df_figuras)} figuras validadas**"))

In [ ]:
# Resumo consolidado para os autores
resumo_final = pd.DataFrame({
    'Componente': [
        'Dataset final',
        'Tabelas exportadas',
        'Figuras validadas',
        'Validação de consistência',
        'Validação de figuras',
        'Metadados da execução'
    ],
    'Status': [
        'Concluído',
        f"{len(tabelas_exportar)} tabelas",
        f"{len(figuras_submissao)} figuras",
        'Preenchido',
        'Validado',
        ' Gerado'
    ],
    'Caminho': [
        DIR_DADOS,
        DIR_TABELAS,
        DIR_GRAFICOS,
        DIR_RESULTADOS,
        DIR_DADOS,
        DIR_RESULTADOS
    ]
})

display(Markdown("## Resumo Final"))
display(resumo_final.style.set_properties(**{'text-align': 'left'}))

# Checklist de verificação pré-submissão
checklist_submissao = pd.DataFrame({
    'Item': [
        'Tabelas em formato .xlsx',
        'Figuras em 600+ DPI',
        'P-valores formatados (p-valor<0,001)',
        'Checklist preenchido',
        'Consistência numérica validada',
        'Metadados de execução salvos',
    ],
    'Concluído': [
        '1', '2', '3', '4', '6', 
        '7'
    ]
})

display(Markdown("## Checklist"))
display(checklist_submissao.style.set_properties(**{'text-align': 'center'}))

# Salvar resumo final
resumo_final.to_csv(
    os.path.join(DIR_RESULTADOS, "resumo_final_submissao.csv"),
    index=False,
    encoding='utf-8'
)

logger.info("="*60)
logger.info("CONCLUÍDO COM SUCESSO")
logger.info("="*60)
logger.info(f"Tabelas: {len(tabelas_exportar)}")
logger.info(f"Figuras: {len(figuras_submissao)}")
logger.info(f"Checklist: Salvo")
logger.info("="*60)

print("\nExportação, Validação e Documentação Final - CONCLUÍDO")
print(f"Diretório de resultados: {os.path.abspath(DIR_RESULTADOS)}")
print(f"Pront ")

In [ ]:
# RESUMO INTEGRADO DAS ANÁLISES ESTRATIFICADAS


display(Markdown("# Resumo Integrado das Análises Estratificadas"))

# Verificar arquivos gerados
arquivos_gerados = {
    'Temporal': os.path.join(DIR_TABELAS, "analise_temporal_por_procedimento.csv"),
    'Custos': os.path.join(DIR_TABELAS, "analise_custos_por_procedimento.csv"),
    'Mapa Contagem': os.path.join(DIR_MAPAS, 'mapa_brasil_contagem_total.png'),
    'Mapa Taxa': os.path.join(DIR_MAPAS, 'mapa_brasil_taxa_100k.png'),
    'Mapa Custo': os.path.join(DIR_MAPAS, 'mapa_brasil_custo_medio.png')
}


# Carregar e exibir resumos
if os.path.exists(arquivos_gerados['Temporal']):
    df_temp = pd.read_csv(arquivos_gerados['Temporal'], sep=';', encoding='utf-8')
    display(Markdown("## Tendências Temporais por Procedimento"))
    display(df_temp[['Procedimento', 'Total_Procedimentos', 'Taxa_Crescimento_Mensal_Pct', 'R2']].style.format({
        'Total_Procedimentos': '{:,.0f}',
        'Taxa_Crescimento_Mensal_Pct': '{:+.2f}%',
        'R2': '{:.3f}'
    }))

if os.path.exists(arquivos_gerados['Custos']):
    df_custos = pd.read_csv(arquivos_gerados['Custos'], sep=';', encoding='utf-8')
    display(Markdown("## Custos por Procedimento"))
    display(df_custos[['Procedimento', 'N', 'Mediana', 'IIQ', 'CV_Pct']].style.format({
        'N': '{:,.0f}',
        'Mediana': 'R$ {:,.2f}',
        'IIQ': 'R$ {:,.2f}',
        'CV_Pct': '{:.1f}%'
    }))

logger.info("Resumo integrado das análises estratificadas concluído")
print(f"\nAnálises estratificadas concluídas")
print(f"Tabelas: {DIR_TABELAS}")
print(f"Mapas: {DIR_MAPAS}")